In [39]:
!pip install pykeen -q

In [74]:
!pip install transformers

In [40]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes

In [41]:
import warnings
warnings.filterwarnings("ignore")

import os, json, csv, time, pickle, torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from collections import defaultdict, deque, Counter
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
import torch, pickle 

In [42]:
import json, os

BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"

# Peek at entities.json structure
with open(f"{BASE}/data/entities/en/entities.json") as f:
    ent_json = json.load(f)

# See what it looks like
first_keys = list(ent_json.keys())[:3]
for k in first_keys:
    print(k, "→", ent_json[k])

# Peek at one txt file
sample_txt = f"{BASE}/data/entities/en/extracts/extracts/Q100.txt"
with open(sample_txt) as f:
    print("\nQ100.txt:\n", f.read()[:300])

Q202042 → {'wiki': 'https://en.wikipedia.org/wiki/Euskaltzaindia', 'description': 'official academic language regulatory institution which watches over the Basque language', 'label': 'Euskaltzaindia'}
Q113400 → {'wiki': 'https://en.wikipedia.org/wiki/Fritz_Glatz', 'description': 'Austrian racing driver', 'label': 'Fritz Glatz'}
Q432602 → {'wiki': 'https://en.wikipedia.org/wiki/David_Hidalgo', 'description': 'American musician', 'label': 'David Hidalgo'}

Q100.txt:
 Boston (UK: , US: ) is the capital and most populous city of the Commonwealth of Massachusetts in the United States, and the 21st most populous city in the United States. The city proper covers 49 square miles (127 km2) with an estimated population of 694,583 in 2018, also making it the most populou


In [43]:
import json, os, pandas as pd

BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"


with open(f"{BASE}/data/entities/en/entities.json") as f:
    ent_json = json.load(f)   # { "Q202042": {"label": "Euskaltzaindia", ...}, ... }

with open(f"{BASE}/data/relations/en/relations.json") as f:
    rel_json = json.load(f)   # same nested structure for relations

def ent(qid): return ent_json.get(qid, {}).get("label", qid)
def rel(pid): return rel_json.get(pid, {}).get("label", pid)

def load_split(split_name):
    path = f"{BASE}/data/triples/codex-s/{split_name}.txt"
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 3:
                h, r, t = parts
                rows.append((ent(h), rel(r), ent(t)))
    return pd.DataFrame(rows, columns=["head", "relation", "tail"])

df_train = load_split("train")
df_test  = load_split("test")
df_valid = load_split("valid")

print(f"Train: {len(df_train)}  Valid: {len(df_valid)}  Test: {len(df_test)}")
print(df_train.head(10))

def read_id_list(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

entity_qids   = sorted(ent_json.keys())
relation_pids = sorted(rel_json.keys())

entity_to_id   = {ent(qid): i for i, qid in enumerate(entity_qids)}
relation_to_id = {rel(pid): i for i, pid in enumerate(relation_pids)}
id_to_entity   = {i: ent(qid) for i, qid in enumerate(entity_qids)}
id_to_relation = {i: rel(pid) for i, pid in enumerate(relation_pids)}

print(f"Entities: {len(entity_to_id)}   Relations: {len(relation_to_id)}")
print(id_to_entity[0], "|", id_to_entity[1], "|", id_to_entity[2])
print(id_to_relation[0], "|", id_to_relation[1])

Train: 32888  Valid: 1827  Test: 1828
                           head                              relation  \
0                Leonhard Euler  languages spoken, written, or signed   
1                 Carl Djerassi                        cause of death   
2                      Colombia                             member of   
3  Aleksey Nikolayevich Tolstoy  languages spoken, written, or signed   
4                   Netherlands                             member of   
5                 Henry Rollins                            occupation   
6                Richard Wagner                            occupation   
7                   Jaden Smith                            occupation   
8               La Toya Jackson                               sibling   
9         John Cameron Mitchell                            occupation   

                                tail  
0                             German  
1                             cancer  
2  International Finance Corporation  
3 

In [44]:
df_train.to_csv("CODEX_S_train.csv",index = None)
df_test.to_csv("CODEX_S_test.csv",index = None)
df_valid.to_csv("CODEX_S_valid.csv",index = None)

In [45]:
!git clone https://github.com/uma-pi1/kge.git

fatal: destination path 'kge' already exists and is not an empty directory.


In [46]:
import sys

# CLEAN everything related to kge
sys.modules.pop("kge", None)

# REMOVE wrong paths
sys.path = [p for p in sys.path if "kge" not in p]

# ADD ONLY THIS
sys.path.insert(0, "/kaggle/working/kge")

In [47]:
import kge
print(kge.__file__)

/kaggle/working/kge/kge/__init__.py


In [48]:
import sys
sys.modules.pop("kge", None)

<module 'kge' from '/kaggle/working/kge/kge/__init__.py'>

In [49]:
file_path = "/kaggle/working/kge/kge/job/__init__.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "search_ax" in line:
        continue  # REMOVE the line entirely
    new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Removed Ax import completely")

Removed Ax import completely


In [50]:
!rm /kaggle/working/kge/kge/job/search_ax.py

rm: cannot remove '/kaggle/working/kge/kge/job/search_ax.py': No such file or directory


In [51]:
file_path = "/kaggle/working/kge/kge/job/__init__.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "search_grash" in line:
        continue  # remove this import
    new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Removed GraSH import")

Removed GraSH import


In [52]:
from kge.model import KgeModel
from kge.util.io import load_checkpoint

print("Import worked")

Import worked


In [53]:
MODEL_PATH = r"/kaggle/input/datasets/aaryaupi/winner-model/codex-s-complex-winner/winner_model.pt"
print(MODEL_PATH)


/kaggle/input/datasets/aaryaupi/winner-model/codex-s-complex-winner/winner_model.pt


In [54]:
file_path = "/kaggle/working/kge/kge/util/io.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "torch.load(checkpoint_file, map_location=\"cpu\")",
    "torch.load(checkpoint_file, map_location=\"cpu\", weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patched torch.load")

Patched torch.load


In [55]:
import os

os.environ["KGE_DATASET_DIR"] = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master/data/triples"

In [56]:
import torch
from kge.config import Config

torch.serialization.add_safe_globals([
    torch.torch_version.TorchVersion,
    Config
])

In [57]:
import sys
import torch

sys.path.insert(0, "/kaggle/working")

from kge.config import Config
from kge.util.io import load_checkpoint

# PyTorch safety fix
torch.serialization.add_safe_globals([
    torch.torch_version.TorchVersion,
    Config
])

# reload checkpoint
checkpoint = load_checkpoint(MODEL_PATH, device="cpu")

print("Checkpoint loaded")

Checkpoint loaded


In [58]:
from pykeen.triples import TriplesFactory

In [59]:
import torch, json, pickle, numpy as np
from pathlib import Path

MODEL_DIR  = "/kaggle/input/datasets/aaryaupi/model-libkge-kaggle-upload/kaggle_upload"
CODEX_BASE = "/kaggle/input/datasets/aaryaupi/codex-dataset/codex-master"

CKPT_PATH     = f"{MODEL_DIR}/winner_model.pt"
ENTITY_IDS    = f"{MODEL_DIR}/entity_ids.del"
RELATION_IDS  = f"{MODEL_DIR}/relation_ids.del"

import torch

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state_dict = ckpt["model"][0]

entity_emb   = state_dict["_base_model._entity_embedder._embeddings.weight"].detach().float()
relation_emb = state_dict["_base_model._relation_embedder._embeddings.weight"].detach().float()

print(f"entity_emb:   {entity_emb.shape}")
print(f"relation_emb: {relation_emb.shape}")
print(f"trained epoch: {ckpt['epoch']}")

# Fix here
best_valid = ckpt["valid_trace"][-1]
print(f"best valid MRR: {best_valid['mean_reciprocal_rank_filtered_with_test']:.4f}")

entity_emb:   torch.Size([2034, 512])
relation_emb: torch.Size([84, 512])
trained epoch: 295
best valid MRR: 0.4742


In [60]:

def load_del(path):
    id_to_qid = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                id_to_qid[int(parts[0])] = parts[1]
    return id_to_qid

id_to_entity_qid   = load_del(ENTITY_IDS)
id_to_relation_pid = load_del(RELATION_IDS)
entity_qid_to_id   = {v: k for k, v in id_to_entity_qid.items()}
relation_pid_to_id = {v: k for k, v in id_to_relation_pid.items()}


with open(f"{CODEX_BASE}/data/entities/en/entities.json")  as f: ent_json = json.load(f)
with open(f"{CODEX_BASE}/data/relations/en/relations.json") as f: rel_json = json.load(f)

def elabel(qid): return ent_json.get(qid, {}).get("label", qid)
def rlabel(pid): return rel_json.get(pid, {}).get("label", pid)

id_to_entity_label   = {i: elabel(q) for i, q in id_to_entity_qid.items()}
id_to_relation_label = {i: rlabel(p) for i, p in id_to_relation_pid.items()}
label_to_entity_id   = {v: k for k, v in id_to_entity_label.items()}
label_to_relation_id = {v: k for k, v in id_to_relation_label.items()}

print(f"Entities:  {len(id_to_entity_label)}")
print(f"Relations: {len(id_to_relation_label)}")
print(f"\nAll relations:")
for i, label in sorted(id_to_relation_label.items()):
    print(f"  {i:3d}: {label}")

Entities:  2034
Relations: 42

All relations:
    0: languages spoken, written, or signed
    1: cause of death
    2: member of
    3: occupation
    4: sibling
    5: diplomatic relation
    6: continent
    7: ethnic group
    8: country of citizenship
    9: employer
   10: educated at
   11: place of birth
   12: influenced by
   13: field of work
   14: genre
   15: record label
   16: instrument
   17: official language
   18: unmarried partner
   19: place of death
   20: country
   21: religion
   22: movement
   23: residence
   24: spouse
   25: member of political party
   26: place of burial
   27: part of
   28: medical condition
   29: child
   30: time period
   31: headquarters location
   32: narrative location
   33: location of formation
   34: head of state
   35: named after
   36: parent organization
   37: notable works
   38: cast member
   39: founded by
   40: country of origin
   41: practiced by


In [61]:
def kvs_to_dict(kvs_index):
    index_of_key = kvs_index._index_of_key   # dict: key -> row_idx
    values        = kvs_index._values         # 1D LongTensor, all values concatenated
    offsets        = kvs_index._values_offset  # 1D LongTensor, len = n_keys + 1

    result = {}
    for key, row_idx in index_of_key.items():
        start = offsets[row_idx].item()
        end   = offsets[row_idx + 1].item()
        result[key] = set(values[start:end].tolist())
    return result

with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f:
    train_sp_to_o = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-train_po_to_s.pckl", "rb") as f:
    train_po_to_s = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_sp_to_o.pckl", "rb") as f:
    test_sp_to_o  = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_po_to_s.pckl", "rb") as f:
    test_po_to_s  = kvs_to_dict(pickle.load(f))

print("Filter indexes loaded")

Filter indexes loaded


In [62]:
with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f:
    idx = pickle.load(f)

# Inspect what's available
print(type(idx))
print([a for a in dir(idx) if not a.startswith("__")])

<class 'kge.indexing.KvsAllIndex'>
['_convert_key_to_correct_type', '_create_index_of_key_dict', '_get_all_impl', '_index_of_key', '_keys', '_values', '_values_of', '_values_offset', 'default_factory', 'get', 'get_all', 'items', 'key_cols', 'keys', 'sort_triples_by_keys', 'value_col', 'values']


In [63]:

DIM = entity_emb.shape[1] // 2  # 256

def _re(e): return e[..., :DIM]
def _im(e): return e[..., DIM:]

def score_triple(s, p, o):
    sr, si = _re(entity_emb[s]), _im(entity_emb[s])
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    or_, oi = _re(entity_emb[o]), _im(entity_emb[o])
    return (torch.sum(sr*pr*or_) - torch.sum(si*pi*or_) +
            torch.sum(sr*pi*oi) + torch.sum(si*pr*oi)).item()

def _scores_sp(s, p):
    sr, si = _re(entity_emb[s]), _im(entity_emb[s])
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    return (_re(entity_emb) @ (sr*pr - si*pi) +
            _im(entity_emb) @ (sr*pi + si*pr))

def _scores_po(p, o):
    pr, pi = _re(relation_emb[p]), _im(relation_emb[p])
    or_, oi = _re(entity_emb[o]), _im(entity_emb[o])
    return (_re(entity_emb) @ (pr*or_ - pi*oi) +
            _im(entity_emb) @ (pr*oi + pi*or_))

def kvs_to_dict(kvs_index):
    index_of_key = kvs_index._index_of_key
    values       = kvs_index._values
    offsets      = kvs_index._values_offset
    result = {}
    for key, row_idx in index_of_key.items():
        start = offsets[row_idx].item()
        end   = offsets[row_idx + 1].item()
        result[key] = set(values[start:end].tolist())
    return result

with open(f"{MODEL_DIR}/index-train_sp_to_o.pckl", "rb") as f: train_sp_to_o = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-train_po_to_s.pckl", "rb") as f: train_po_to_s = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_sp_to_o.pckl",  "rb") as f: test_sp_to_o  = kvs_to_dict(pickle.load(f))
with open(f"{MODEL_DIR}/index-test_po_to_s.pckl",  "rb") as f: test_po_to_s  = kvs_to_dict(pickle.load(f))
print("Filter indexes loaded")


def evaluate(split="test"):
    path = f"{CODEX_BASE}/data/triples/codex-s/{split}.txt"
    rr_list, hits1_list, hits10_list = [], [], []

    with open(path) as f:
        triples = [l.strip().split("\t") for l in f if l.strip()]

    for h_qid, r_pid, t_qid in triples:
        s = entity_qid_to_id.get(h_qid)
        p = relation_pid_to_id.get(r_pid)
        o = entity_qid_to_id.get(t_qid)
        if None in (s, p, o):
            continue

        scores = _scores_sp(s, p).clone()
        known  = train_sp_to_o.get((s, p), set()) | test_sp_to_o.get((s, p), set())
        known.discard(o)
        for k in known:
            scores[k] = float("-inf")

        rank = (scores > scores[o]).sum().item() + 1
        rr_list.append(1.0 / rank)
        hits1_list.append(1 if rank == 1 else 0)
        hits10_list.append(1 if rank <= 10 else 0)

    mrr    = float(np.mean(rr_list))
    hits1  = float(np.mean(hits1_list))
    hits10 = float(np.mean(hits10_list))
    print(f"\n── {split.upper()} SET ──────────────────────────────")
    print(f"  MRR:     {mrr:.4f}  (paper: 0.465)")
    print(f"  Hits@1:  {hits1:.4f}  (paper: 0.372)")
    print(f"  Hits@10: {hits10:.4f}  (paper: 0.646)")
    return mrr, hits1, hits10


evaluate("test")

Filter indexes loaded

── TEST SET ──────────────────────────────
  MRR:     0.5759  (paper: 0.465)
  Hits@1:  0.4294  (paper: 0.372)
  Hits@10: 0.8714  (paper: 0.646)


(0.575868337490121, 0.42943107221006566, 0.8714442013129103)

In [64]:
def build_graph_from_df(df):
    graph = defaultdict(list)
    for _, row in df.iterrows():
        graph[row["head"]].append((row["relation"], row["tail"]))
    return graph

graph = build_graph_from_df(df_train)
print(f"Graph nodes: {len(graph)}")

Graph nodes: 1702


In [65]:
EMBS_CACHE = None

def get_entity_embeddings():
    """Return real-valued entity embedding matrix for similarity."""
    embs = entity_emb.float()
    # ComplEx: collapse complex dims by taking magnitude per pair
    re = embs[:, :DIM]
    im = embs[:, DIM:]
    return torch.sqrt(re**2 + im**2)   # shape: (n_entities, DIM)

# ── Subgraph helpers ──────────────────────────────────────────
def extract_subgraph(entity, graph, hops=2, max_triples=150):
    subgraph = []
    visited  = {entity}
    queue    = deque([(entity, 0)])
    while queue and len(subgraph) < max_triples:
        node, depth = queue.popleft()
        if depth >= hops:
            continue
        for rel_label, tail in graph.get(node, []):
            if len(subgraph) >= max_triples:
                break
            subgraph.append((node, rel_label, tail))
            if tail not in visited:
                visited.add(tail)
                queue.append((tail, depth + 1))
    return subgraph

def focused_subgraph(entities, graph, hops=2, max_triples=100, query_relation=None):
    entity_set  = set(entities)
    all_triples = []
    seen        = set()
    for ent_node in entities:
        for triple in extract_subgraph(ent_node, graph, hops, max_triples):
            if triple not in seen:
                seen.add(triple)
                all_triples.append(triple)
    if len(all_triples) <= max_triples:
        return all_triples
    tier1, tier2, tier3, tier4 = [], [], [], []
    for h, r, t in all_triples:
        if query_relation and r == query_relation:       tier1.append((h, r, t))
        elif h in entity_set and t in entity_set:        tier2.append((h, r, t))
        elif h in entity_set or  t in entity_set:        tier3.append((h, r, t))
        else:                                             tier4.append((h, r, t))
    return (tier1 + tier2 + tier3 + tier4)[:max_triples]

def hop_classifier(head, tail, graph, target_relation=None):
    for relation, t in graph.get(head, []):
        if t == tail:
            if target_relation is None or relation == target_relation:
                return "single", 1, relation
    direct_wrong = [r for r, t in graph.get(head, []) if t == tail]
    if direct_wrong:
        return "multi", 1.5, f"direct but via {direct_wrong[0]}"
    paths_found = []
    for r1, mid in graph.get(head, []):
        for r2, t2 in graph.get(mid, []):
            if t2 == tail:
                paths_found.append(f"{head}-{r1}->{mid}-{r2}->{tail}")
    if paths_found:
        return "multi", 2, paths_found[0]
    return "none", 99, []

In [66]:
def get_entity_relations(entity, df):
    head_rels = set(df[df["head"] == entity]["relation"])
    tail_rels = set(df[df["tail"] == entity]["relation"])
    return head_rels | tail_rels

def similarity_summary(entity, k=5):
    e_id  = label_to_entity_id[entity]
    e_vec = EMBS_CACHE[e_id]
    sims  = F.cosine_similarity(e_vec.unsqueeze(0), EMBS_CACHE).detach().cpu().numpy()
    ranked = np.argsort(sims)[::-1]
    entity_rels = get_entity_relations(entity, df_train)
    results = []
    for idx in ranked:
        name = id_to_entity_label[idx]
        if name == entity:
            continue
        score = sims[idx]
        neighbor_rels = get_entity_relations(name, df_train)
        shared    = len(entity_rels & neighbor_rels)
        total     = len(entity_rels | neighbor_rels)
        rel_score = shared / total if total > 0 else 0.0
        results.append((name, score, shared, rel_score))
        if len(results) == k:
            break
    parts   = [f"{n}(sim={s:.2f},shared={sh},rel={rs:.2f})"
               for n, s, sh, rs in results]
    summary = f"{entity} most similar to: {', '.join(parts)}"
    return summary, results

In [67]:
def get_full_ranking_filtered_batched(head, relation, true_tail):
    """
    Score ALL tail candidates using raw ComplEx embeddings.
    Replaces winner_model.score_hrt() entirely.
    """
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    o = label_to_entity_id.get(true_tail)

    if None in (s, p, o):
        raise ValueError(f"Unknown entity/relation: {head}, {relation}, {true_tail}")

    with torch.no_grad():
        all_scores = _scores_sp(s, p).cpu()   # shape: (n_entities,)

    # exclude head itself from ranking
    head_id = label_to_entity_id[head]
    all_scores[head_id] = float("-inf")

    scores_list = all_scores.tolist()
    ranked = sorted(
        [(id_to_entity_label[i], scores_list[i]) for i in range(len(scores_list))
         if scores_list[i] != float("-inf")],
        key=lambda x: -x[1]
    )

    ranked_entities = [e for e, _ in ranked]
    true_rank       = ranked_entities.index(true_tail) + 1

    return {
        "predicted":       ranked[0][0],
        "predicted_score": ranked[0][1],
        "true_tail":       true_tail,
        "true_score":      scores_list[o],
        "true_rank":       true_rank,
        "full_ranking":    [(e, round(sc, 3)) for e, sc in ranked],
    }


In [68]:
def get_full_ranking_filtered(head, relation, true_tail, model):
    """
    Same as before but filters self-loop at rank 1.
    In real routing you'd never route a problem to itself.
    """
    h_id = entity_to_id[head]
    r_id = relation_to_id[relation]
    
    all_scores = []
    for i in range(len(entity_to_id)):
        # skip self-prediction
        if id_to_entity[i] == head:
            continue
        score = model.score_hrt(
            torch.tensor([[h_id, r_id, i]])
        ).item()
        all_scores.append((id_to_entity[i], score, i))
    
    all_scores.sort(key=lambda x: -x[1])
    
    ranked_entities = [e for e, s, i in all_scores]
    true_rank       = ranked_entities.index(true_tail) + 1
    predicted       = all_scores[0][0]
    
    return {
        "predicted":       predicted,
        "predicted_score": all_scores[0][1],
        "true_tail":       true_tail,
        "true_score":      [s for e,s,i in all_scores 
                           if e == true_tail][0],
        "true_rank":       true_rank,
        "full_ranking":    [(e, round(s,3)) 
                           for e,s,i in all_scores],
    }

In [69]:
def build_type_constraints(df):
    rel_to_tail_counts = defaultdict(lambda: defaultdict(int))
    rel_to_head_counts = defaultdict(lambda: defaultdict(int))
    rel_to_pair_counts = defaultdict(lambda: defaultdict(int))
    for _, row in df.iterrows():
        h, r, t = row["head"], row["relation"], row["tail"]
        rel_to_tail_counts[r][t] += 1
        rel_to_head_counts[r][h] += 1
        rel_to_pair_counts[r][(h, t)] += 1
    rel_head_to_ranked_tails = defaultdict(dict)
    for r, pair_counts in rel_to_pair_counts.items():
        head_to_tails = defaultdict(dict)
        for (h, t), count in pair_counts.items():
            head_to_tails[h][t] = count
        for h, tail_counts in head_to_tails.items():
            total = sum(tail_counts.values())
            rel_head_to_ranked_tails[(r, h)] = {
                t: round(c / total, 4)
                for t, c in sorted(tail_counts.items(), key=lambda x: -x[1])
            }
    rel_to_tail_dist = {}
    for r, tail_counts in rel_to_tail_counts.items():
        total = sum(tail_counts.values())
        rel_to_tail_dist[r] = {
            t: round(c / total, 4)
            for t, c in sorted(tail_counts.items(), key=lambda x: -x[1])
        }
    return {
        "rel_head_to_ranked_tails": dict(rel_head_to_ranked_tails),
        "rel_to_tail_dist":         rel_to_tail_dist,
        "rel_to_tail_counts":       dict(rel_to_tail_counts),
        "rel_to_head_counts":       dict(rel_to_head_counts),
    }

def get_type_constraint_signal(head, relation, true_tail, predicted, constraints):
    rh_tails       = constraints["rel_head_to_ranked_tails"]
    tail_dist      = constraints["rel_to_tail_dist"]
    specific       = rh_tails.get((relation, head), {})
    general        = tail_dist.get(relation, {})
    type_fit_true  = specific.get(true_tail) or general.get(true_tail, 0.0)
    type_fit_pred  = specific.get(predicted) or general.get(predicted, 0.0)
    expected_tails = list(specific.keys())[:5] or list(general.keys())[:5]
    tail_counts    = constraints["rel_to_tail_counts"]
    true_rels = {r for r, tails in tail_counts.items() if true_tail in tails}
    pred_rels = {r for r, tails in tail_counts.items() if predicted  in tails}
    return {
        "type_fit_true":  type_fit_true,
        "type_fit_pred":  type_fit_pred,
        "type_gap":       round(type_fit_true - type_fit_pred, 4),
        "expected_tails": expected_tails,
        "only_true_has":  sorted(true_rels - pred_rels),
        "only_pred_has":  sorted(pred_rels - true_rels),
        "shared_rels":    sorted(true_rels & pred_rels),
    }


In [70]:
def failure_summary(head, relation, true_tail, predicted_tail, constraints):
    """Uses score_triple() with raw embeddings — no model object needed."""
    s  = label_to_entity_id[head]
    p  = label_to_relation_id[relation]
    o  = label_to_entity_id[true_tail]
    p2 = label_to_entity_id[predicted_tail]

    score_true = score_triple(s, p, o)
    score_pred = score_triple(s, p, p2)

    sig = get_type_constraint_signal(head, relation, true_tail, predicted_tail, constraints)
    summary = (
        f"Model predicted '{predicted_tail}' (score={score_pred:.3f}) "
        f"over '{true_tail}' (score={score_true:.3f}). "
        f"Type fit: '{true_tail}'={sig['type_fit_true']:.3f} vs "
        f"'{predicted_tail}'={sig['type_fit_pred']:.3f} for '{relation}'. "
        f"Expected tails: {', '.join(sig['expected_tails'][:3])}. "
        f"Only '{true_tail}' in: {', '.join(sig['only_true_has'][:5]) or 'none'}. "
        f"Only '{predicted_tail}' in: {', '.join(sig['only_pred_has'][:5]) or 'none'}."
    )
    return summary, {
        "score_true":     score_true,
        "score_pred":     score_pred,
        "shared":         set(sig["shared_rels"]),
        "only_true":      set(sig["only_true_has"]),
        "only_pred":      set(sig["only_pred_has"]),
        "type_fit_true":  sig["type_fit_true"],
        "type_fit_pred":  sig["type_fit_pred"],
        "type_gap":       sig["type_gap"],
        "expected_tails": sig["expected_tails"],
    }


In [71]:
@torch.no_grad()
def batch_rank_triples(tuples_batch: list, hard_threshold: int) -> list:
    valid = []
    for i, (h, r, t) in enumerate(tuples_batch):
        s = label_to_entity_id.get(h)
        p = label_to_relation_id.get(r)
        o = label_to_entity_id.get(t)
        if None in (s, p, o):
            valid.append((i, None))
        else:
            valid.append((i, (s, p, o, h, t)))

    good = [(i, d) for i, d in valid if d is not None]
    out  = [None] * len(tuples_batch)
    if not good:
        return out

    s_ids = torch.tensor([d[0] for _, d in good], device=DEVICE)
    p_ids = torch.tensor([d[1] for _, d in good], device=DEVICE)

    s_re = ENT_RE[s_ids]
    s_im = ENT_IM[s_ids]
    p_re = relation_emb_gpu[p_ids, :DIM]
    p_im = relation_emb_gpu[p_ids, DIM:]

    scores = (s_re * p_re - s_im * p_im) @ ENT_RE.T \
           + (s_re * p_im + s_im * p_re) @ ENT_IM.T

    for row_pos, (orig_idx, d) in enumerate(good):
        s, p, o, h_label, t_label = d
        row = scores[row_pos].clone()
        row[s] = float("-inf")

        ranked_ids  = torch.argsort(row, descending=True).cpu()
        scores_cpu  = row.cpu().numpy()
        ranked_list = ranked_ids.tolist()
        true_rank   = ranked_list.index(o) + 1

        out[orig_idx] = {
            "predicted":       id_to_entity_label[ranked_list[0]],
            "predicted_score": float(scores_cpu[ranked_list[0]]),
            "true_tail":       t_label,
            "true_score":      float(scores_cpu[o]),
            "true_rank":       true_rank,
            "full_ranking":    [(id_to_entity_label[i], round(float(scores_cpu[i]), 3))
                                for i in ranked_list[:200]],
            "hard_failure":    bool(true_rank > hard_threshold),
        }

    return out

In [72]:
import os, json, time
import torch
import torch.nn.functional as F
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

entity_emb_gpu   = entity_emb.to(DEVICE).float()
relation_emb_gpu = relation_emb.to(DEVICE).float()
N_ENT = entity_emb_gpu.shape[0]
DIM   = entity_emb_gpu.shape[1] // 2

ENT_RE = entity_emb_gpu[:, :DIM].contiguous()
ENT_IM = entity_emb_gpu[:, DIM:].contiguous()

_mag      = torch.sqrt(ENT_RE**2 + ENT_IM**2)
EMBS_NORM = F.normalize(_mag, dim=-1).contiguous()
EMBS_CACHE = _mag.cpu()

print("Building entity-relation cache …")
ent_to_rels: dict[str, set] = {}
for _, row in df_train.iterrows():
    h, r, t = row["head"], row["relation"], row["tail"]
    ent_to_rels.setdefault(h, set()).add(r)
    ent_to_rels.setdefault(t, set()).add(r)
print(f"  cached {len(ent_to_rels)} entities")

# Pre-cache head → relation → list of tails from training
ent_rel_to_tails: dict[tuple, list] = {}
for _, row in df_train.iterrows():
    key = (row["head"], row["relation"])
    ent_rel_to_tails.setdefault(key, []).append(row["tail"])
print(f"  cached {len(ent_rel_to_tails)} (entity, relation) pairs")

constraints = build_type_constraints(df_train)


def is_hard_failure(ranking, head, relation, constraints, hop_type="multi"):
    if ranking["true_rank"] == 1: return False
    if hop_type == "none":        return False
    return ranking["true_rank"] <= 12


def reclassify_hard(records, constraints):
    reclassified = 0
    for r in records:
        fake_ranking = {
            "true_rank":       r["true_rank"],
            "true_tail":       r["tail"],
            "predicted":       r["predicted"],
            "true_score":      r["score_true"],
            "predicted_score": r["score_predicted"],
        }
        new_hard = is_hard_failure(
            fake_ranking,
            r["head"],
            r["relation"],
            constraints,
            hop_type=r.get("hop_type", "multi")
        )
        if new_hard != r["hard_failure"]:
            r["hard_failure"] = new_hard
            reclassified += 1
    print(f"Reclassified {reclassified} records")
    return records


@torch.no_grad()
def batch_similarity_summary(entity_labels: list, k: int = 5) -> list:
    ids  = torch.tensor([label_to_entity_id[e] for e in entity_labels], device=DEVICE)
    vecs = EMBS_NORM[ids]
    sim_matrix = vecs @ EMBS_NORM.T
    topk_vals, topk_ids = torch.topk(sim_matrix, k + 1, dim=-1)
    topk_ids_cpu  = topk_ids.cpu().numpy()
    topk_vals_cpu = topk_vals.cpu().numpy()
    results = []
    for b_idx, entity in enumerate(entity_labels):
        e_rels  = ent_to_rels.get(entity, set())
        entries = []
        for j in range(topk_ids_cpu.shape[1]):
            idx  = int(topk_ids_cpu[b_idx, j])
            name = id_to_entity_label[idx]
            if name == entity:
                continue
            score  = float(topk_vals_cpu[b_idx, j])
            n_rels = ent_to_rels.get(name, set())
            shared = len(e_rels & n_rels)
            total  = len(e_rels | n_rels)
            rel_sc = shared / total if total > 0 else 0.0
            entries.append((name, score, shared, rel_sc))
            if len(entries) == k:
                break
        parts   = [f"{n}(sim={s:.2f},shared={sh},rel={rs:.2f})" for n, s, sh, rs in entries]
        summary = f"{entity} most similar to: {', '.join(parts)}"
        results.append((summary, entries))
    return results


def _cpu_record(h, r, t, ranking):
    sg = focused_subgraph(
        [h, t, ranking["predicted"]],
        graph, hops=3, max_triples=150, query_relation=r,
    )
    hop_type, hops, _ = hop_classifier(h, t, graph, target_relation=r)
    fail_sum, fail_raw = failure_summary(h, r, t, ranking["predicted"], constraints)
    return sg, hop_type, int(hops), fail_sum, fail_raw


def preprocess_all_triples_gpu(df, split_name, batch_size=128, cpu_workers=4):
    records        = []
    n_entities     = len(label_to_entity_id)
    hard_threshold = max(10, int(n_entities * 0.15))
    rows           = [(r["head"], r["relation"], r["tail"]) for _, r in df.iterrows()]
    total          = len(rows)
    t0             = time.time()

    for batch_start in range(0, total, batch_size):
        batch = rows[batch_start: batch_start + batch_size]
        valid = [
            (h, r, t) for h, r, t in batch
            if h in label_to_entity_id
            and t in label_to_entity_id
            and r in label_to_relation_id
        ]
        if not valid:
            continue

        rankings = batch_rank_triples(valid, hard_threshold)

        heads = [h for h, r, t in valid]
        tails = [t for h, r, t in valid]
        sims  = batch_similarity_summary(heads + tails, k=5)
        sim_heads = sims[:len(heads)]
        sim_tails = sims[len(heads):]

        def _work(args):
            i, (h, r, t) = args
            if rankings[i] is None:
                return i, None
            try:
                return i, _cpu_record(h, r, t, rankings[i])
            except Exception as e:
                print(f"\n  [CPU ERR] {h} | {r} | {t}: {e}")
                return i, None

        cpu_out = {}
        with ThreadPoolExecutor(max_workers=cpu_workers) as ex:
            for i, res in ex.map(_work, enumerate(valid)):
                cpu_out[i] = res

        for i, (h, r, t) in enumerate(valid):
            rk  = rankings[i]
            cpu = cpu_out.get(i)
            if rk is None or cpu is None:
                continue
            sg, hop_type, hops, fail_sum, fail_raw = cpu

            records.append({
                "split":            split_name,
                "head":             h,
                "relation":         r,
                "tail":             t,
                "true_rank":        int(rk["true_rank"]),
                "predicted":        rk["predicted"],
                "score_true":       float(rk["true_score"]),
                "score_predicted":  float(rk["predicted_score"]),
                "top5":             rk["full_ranking"][:15],
                "hop_type":         hop_type,
                "hop_count":        hops,
                "sim_head":         sim_heads[i][0],
                "sim_tail":         sim_tails[i][0],
                "fail_summary":     fail_sum,
                "subgraph":         sg,
                "shared_relations": list(fail_raw["shared"]),
                "only_tail_has":    list(fail_raw["only_true"]),
                "only_pred_has":    list(fail_raw["only_pred"]),
                "hard_failure":     is_hard_failure(
                                        rk, h, r, constraints,
                                        hop_type=hop_type),
                "known_tails":      ent_rel_to_tails.get((h, r), []),
            })

        done    = min(batch_start + batch_size, total)
        elapsed = time.time() - t0
        rate    = done / elapsed if elapsed > 0 else 0
        eta     = (total - done) / rate if rate > 0 else 0
        print(f"  {split_name}: {done}/{total}  "
              f"records={len(records)}  "
              f"{rate:.1f} rows/s  ETA {eta/60:.1f}min", end="\r")

    print()
    return records


PREPROCESSED_TRAIN = "/kaggle/working/CODEX_S_preprocessed_train.json"
PREPROCESSED_TEST  = "/kaggle/working/CODEX_S_preprocessed_test.json"

for f in [PREPROCESSED_TRAIN, PREPROCESSED_TEST]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")


t0 = time.time()
print("Preprocessing train …")
train_records = preprocess_all_triples_gpu(df_train, "train", batch_size=128)
print("Preprocessing test …")
test_records  = preprocess_all_triples_gpu(df_test,  "test",  batch_size=128)
with open(PREPROCESSED_TRAIN, "w") as f: json.dump(train_records, f, indent=2)
with open(PREPROCESSED_TEST,  "w") as f: json.dump(test_records,  f, indent=2)
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

if EMBS_CACHE is None:
    EMBS_CACHE = _mag.cpu()

sample = test_records[0]
print(f"\nVerification — top5 length: {len(sample['top5'])}  ← must be 15")
print(f"true_rank of sample: {sample['true_rank']}")
print(f"Correct answer visible: {sample['true_rank'] <= len(sample['top5'])}")
print(f"known_tails sample: {sample['known_tails'][:5]}")

Device: cuda  |  GPU: Tesla T4
Building entity-relation cache …
  cached 2034 entities
  cached 10465 (entity, relation) pairs
Preprocessing train …
  train: 32888/32888  records=32888  390.2 rows/s  ETA 0.0min
Preprocessing test …
  test: 1828/1828  records=1828  385.8 rows/s  ETA 0.0min

Done in 1.8 min

Verification — top5 length: 15  ← must be 15
true_rank of sample: 1
Correct answer visible: True
known_tails sample: []


In [106]:
# ════════════════════════════════════════════════════════
# BLOCK 1 — LOAD DATA
# Changes from UMLS:
#   - Different file paths
#   - agent_results comes from val_hard_results.json
#   - preprocessed_index uses head/relation/tail keys
# ════════════════════════════════════════════════════════

import json, random, numpy as np, torch
from collections import defaultdict, Counter
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MAX_K        = 100
W_KGE        = 0.5
W_AGENT      = 0.3
W_OVERLAP    = 0.2
MAX_LENGTH   = 512

# ── Load agent results from your val_hard run ─────────────────────────────────
with open("/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/val_hard_results (23).json") as f:
    agent_results = json.load(f)

# ── Load preprocessed records ─────────────────────────────────────────────────
with open("/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/CODEX_S_preprocessed_train (2).json") as f:
    train_data = json.load(f)
with open("/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/CODEX_S_preprocessed_test (2).json") as f:
    test_data = json.load(f)

preprocessed_index = {
    (r["head"], r["relation"], r["tail"]): r
    for r in train_data + test_data
}

print(f"Agent results : {len(agent_results)}")
print(f"Train records : {len(train_data)}")
print(f"Test records  : {len(test_data)}")
print(f"Index size    : {len(preprocessed_index)}")


# ── Agent confidence lookup — identical to UMLS ───────────────────────────────
def load_agent_confidence(agent_results):
    lookup = {}
    for r in agent_results:
        agg       = r.get("aggregator", {})
        chosen    = agg.get("chosen_agent", "B")
        agent_out = r.get("agent_a" if chosen == "A" else "agent_b", {})
        conf      = float(agent_out.get("confidence", 0.5))
        t = r.get("triple", "")
        if t.startswith("(") and t.endswith(")"):
            parts = [p.strip() for p in t[1:-1].split(",")]
            if len(parts) == 3:
                lookup[f"{parts[0]}|{parts[1]}|{parts[2]}"] = conf
    print(f"Agent confidence lookup: {len(lookup)} triples")
    return lookup

agent_conf_lookup = load_agent_confidence(agent_results)


# ════════════════════════════════════════════════════════
# BLOCK 2 — ENTITY PROFILES AND KGE SCORING
# Changes from UMLS:
#   - entity_rel_profile uses ent_to_rels (already built)
#   - KGE scoring uses _scores_sp not model.score_hrt
#   - Add known_tails lookup (CoDEx-S specific)
# ════════════════════════════════════════════════════════

# ent_to_rels already built in your GPU setup cell
# maps entity → set of relations it participates in
# No rebuild needed — just reference it directly

# all_entities for random sampling
all_entities = list(label_to_entity_id.keys())

@torch.no_grad()
def get_kge_scores_for_query_codex(head, relation):
    """
    Score all entities as candidates for (head, relation, ?)
    Uses _scores_sp vectorised GPU call — much faster than UMLS
    which called score_hrt in batches
    """
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    if None in (s, p):
        return {}

    scores_gpu = (
        (ENT_RE[s] * relation_emb_gpu[p, :DIM] -
         ENT_IM[s] * relation_emb_gpu[p, DIM:]) @ ENT_RE.T +
        (ENT_RE[s] * relation_emb_gpu[p, DIM:] +
         ENT_IM[s] * relation_emb_gpu[p, :DIM]) @ ENT_IM.T
    )
    scores_gpu[s] = float("-inf")
    scores_cpu    = scores_gpu.cpu().numpy()

    return {
        id_to_entity_label[i]: float(scores_cpu[i])
        for i in range(len(scores_cpu))
        if id_to_entity_label[i] != head
    }

def profile_similarity(e1, e2):
    """Jaccard on relational profiles — same as UMLS"""
    p1 = ent_to_rels.get(e1, set())
    p2 = ent_to_rels.get(e2, set())
    if not p1 or not p2:
        return 0.0
    return len(p1 & p2) / len(p1 | p2)

def type_fit(entity, relation):
    dist = constraints["rel_to_tail_dist"].get(relation, {})
    return dist.get(entity, 0.0)


# ════════════════════════════════════════════════════════
# BLOCK 3 — TEXT INPUT BUILDER
# Changes from UMLS:
#   - Adds [TRAINING EVIDENCE] block (CoDEx-S specific)
#   - known_tails shows what training already knows
#   - in_training flag is key discriminating feature
#   - Everything else identical to UMLS format
# ════════════════════════════════════════════════════════

def build_text_input_codex(head, relation, candidate,
                            features, true_tail=None):
    # ← accepts features dict, not kge_score float
    f         = features
    sig       = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = sig.get(candidate, 0.0)
    ranked    = list(sig.keys())
    type_rank = ranked.index(candidate) + 1 if candidate in ranked else 99
    expected  = ", ".join(ranked[:3]) if ranked else "none"
    cand_rels = sorted(ent_to_rels.get(candidate, set()))[:5]

    known     = ent_rel_to_tails.get((head, relation), [])
    known_str = ", ".join(known[:5]) if known else "none"
    in_train  = int(candidate in known)

    text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{candidate}\n\n"
        f"[TYPE CONSTRAINTS]\n"
        f"type_fit = {tf:.4f}\n"
        f"type_rank = {type_rank}\n"
        f"expected_tails = {expected}\n\n"
        f"[KGE SIGNALS]\n"
        f"kge_score = {f.get('kge_score', 0.0):.3f}\n\n"
        f"[TRAINING EVIDENCE]\n"
        f"known_tails = {known_str}\n"
        f"in_training = {in_train}\n\n"
        f"[RELATIONAL PROFILE]\n"
        f"candidate_appears_in = "
        f"{', '.join(cand_rels) if cand_rels else 'none'}\n"
    )


    return text


# ════════════════════════════════════════════════════════
# TRIPLE PARSER — handles commas inside relation names
# ════════════════════════════════════════════════════════

def parse_triple_string(triple_str, true_tail, known_relations):
    inner = triple_str.strip()
    if inner.startswith("(") and inner.endswith(")"):
        inner = inner[1:-1]
    anchor = f", {true_tail}"
    if not inner.endswith(anchor):
        return None, None
    head_and_relation = inner[:-len(anchor)]
    for rel in sorted(known_relations, key=len, reverse=True):
        if head_and_relation.endswith(f", {rel}"):
            head = head_and_relation[:-len(f", {rel}")]
            return head, rel
    return None, None


# ════════════════════════════════════════════════════════
# BLOCK 4 — NEGATIVE SAMPLING
# Changes from UMLS:
#   - Adds "in_training" negative type (CoDEx-S specific)
#   - These are entities already known as tails for this
#     (head, relation) pair — the exact confusion the
#     model makes (e.g. singer when answer is recording artist)
#   - type_confusable same as UMLS
#   - random same as UMLS
# ════════════════════════════════════════════════════════

def sample_negatives_codex(head, relation, true_tail,
                            kge_scores,
                            n_model=3, n_type=2,
                            n_intrain=1, n_random=1):
    """
    Four negative types for CoDEx-S:

    model_topk:   what ComplEx ranked highly but wrong
                  hardest — model already confident

    type_confusable: non-zero type_fit but wrong
                  medium hard — plausible slot fillers

    in_training:  entities already known as tails for
                  this (head, relation) in training data
                  CoDEx-S specific — these are what the
                  model confuses because they appear in
                  training (singer vs recording artist)

    random:       pure random for diversity
    """
    negatives = []
    seen      = {true_tail, head}

    # ── model_topk ────────────────────────────────────────────────────────────
    model_ranked = sorted(
        [(e, s) for e, s in kge_scores.items() if e not in seen],
        key=lambda x: -x[1]
    )
    for entity, score in model_ranked[:n_model]:
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "model_topk",
            "features": {
                "kge_score":       score,
                "type_fit":        type_fit(entity, relation),
                "profile_sim":     round(profile_similarity(entity, true_tail), 4),
                "jaccard_overlap": round(profile_similarity(entity, true_tail), 4),
                "in_training":     int(entity in ent_rel_to_tails.get((head, relation), [])),
            }
        })
        seen.add(entity)

    # ── in_training (CoDEx-S specific) ───────────────────────────────────────
    # These are the hardest negatives for CoDEx-S because
    # the model has literally seen them as correct answers
    # for similar queries
    known_tails = [
        e for e in ent_rel_to_tails.get((head, relation), [])
        if e not in seen
    ]
    for entity in known_tails[:n_intrain]:
        score = kge_scores.get(entity, -99.0)
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "in_training",
            "features": {
                "kge_score":       score,
                "type_fit":        type_fit(entity, relation),
                "profile_sim":     round(profile_similarity(entity, true_tail), 4),
                "jaccard_overlap": round(profile_similarity(entity, true_tail), 4),
                "in_training":     1,
            }
        })
        seen.add(entity)

    # ── type_confusable ───────────────────────────────────────────────────────
    rel_dist         = constraints["rel_to_tail_dist"].get(relation, {})
    type_candidates  = [
        e for e in rel_dist
        if e not in seen and rel_dist[e] > 0.0
    ]
    type_candidates.sort(key=lambda e: -rel_dist[e])
    for entity in type_candidates[:n_type]:
        score = kge_scores.get(entity, -99.0)
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "type_confusable",
            "features": {
                "kge_score":       score,
                "type_fit":        rel_dist[entity],
                "profile_sim":     round(profile_similarity(entity, true_tail), 4),
                "jaccard_overlap": round(profile_similarity(entity, true_tail), 4),
                "in_training":     int(entity in ent_rel_to_tails.get((head, relation), [])),
            }
        })
        seen.add(entity)

    # ── random ────────────────────────────────────────────────────────────────
    pool = [e for e in all_entities if e not in seen]
    for entity in random.sample(pool, min(n_random, len(pool))):
        score = kge_scores.get(entity, -99.0)
        negatives.append({
            "entity":   entity,
            "label":    0,
            "neg_type": "random",
            "features": {
                "kge_score":       score,
                "type_fit":        type_fit(entity, relation),
                "profile_sim":     round(profile_similarity(entity, true_tail), 4),
                "jaccard_overlap": round(profile_similarity(entity, true_tail), 4),
                "in_training":     0,
            }
        })

    return negatives


# ════════════════════════════════════════════════════════
# BLOCK 5 — SOFT LABELS
# Changes from UMLS:
#   - in_training negatives get higher soft_label
#     because they are genuinely confusable
#     (model has seen them as correct answers before)
#   - Everything else identical to UMLS
# ════════════════════════════════════════════════════════

def compute_soft_labels_codex(candidates, agent_conf):
    negs = [c for c in candidates if c["label"] == 0]
    if not negs:
        return candidates

    kge_vals  = [c["features"].get("kge_score", -99.0) for c in negs]
    type_vals = [c["features"].get("type_fit",   0.0)  for c in negs]
    kge_min,  kge_max  = min(kge_vals), max(kge_vals)
    type_min, type_max = min(type_vals), max(type_vals)
    kge_r  = max(kge_max  - kge_min,  1e-6)
    type_r = max(type_max - type_min, 1e-6)

    for c in candidates:
        if c["label"] == 1:
            c["soft_label"] = 1.0
            continue
        if c.get("neg_type") == "random":
            c["soft_label"] = 0.0
            continue

        kge_n  = (c["features"].get("kge_score", -99.0) - kge_min) / kge_r
        type_n = (c["features"].get("type_fit",   0.0)  - type_min) / type_r
        pen    = (1.0 - agent_conf) * W_AGENT

        soft = W_KGE * kge_n + W_OVERLAP * type_n + pen

        # CoDEx-S specific — boost in_training negatives
        # they are harder because model has literally seen them
        if c.get("neg_type") == "in_training":
            soft = min(soft + 0.15, 0.89)

        c["soft_label"] = round(min(max(soft, 0.0), 0.89), 4)

    return candidates


# ════════════════════════════════════════════════════════
# BLOCK 6 — MAIN BUILDER
# Changes from UMLS:
#   - Uses get_kge_scores_for_query_codex
#   - Passes known_tails to context
#   - Everything else identical
# ════════════════════════════════════════════════════════

def build_full_codex_bert_dataset(agent_results, output_path,
                                   n_model=3, n_type=2,
                                   n_intrain=1, n_random=1,
                                   n_noise=2):
    bert_records = []
    skipped      = 0
    kge_cache    = {}
    stats        = {"correct": 0, "wrong": 0, "parse_fail": 0,
                    "index_miss": 0, "noise_added": 0}

    for i, result in enumerate(agent_results):

        # ── Use ALL results, not just correct+resolved ────────────────────────
        true_tail = result.get("true_tail", "")
        if not true_tail:
            skipped += 1
            continue

        # ── Parse with comma-safe parser ──────────────────────────────────────
        head, relation = parse_triple_string(
            result["triple"],
            true_tail,
            label_to_relation_id.keys()
        )
        if head is None or relation is None:
            print(f"  [PARSE FAIL] {result['triple']}")
            stats["parse_fail"] += 1
            skipped += 1
            continue

        pre = preprocessed_index.get((head, relation, true_tail))
        if pre is None:
            stats["index_miss"] += 1
            skipped += 1
            continue

        is_correct = result.get("final_correct", False)
        agent_pred = result.get("aggregator", {}).get("final_answer", "")
        agent_conf = agent_conf_lookup.get(
            f"{head}|{relation}|{true_tail}", 0.5
        )

        cache_key = (head, relation)
        if cache_key not in kge_cache:
            kge_cache[cache_key] = get_kge_scores_for_query_codex(
                head, relation
            )
        kge_scores = kge_cache[cache_key]
        true_kge   = kge_scores.get(true_tail, pre["score_true"])

        # ── Positive ──────────────────────────────────────────────────────────
        pos_features = {
            "kge_score":       true_kge,
            "type_fit":        type_fit(true_tail, relation),
            "profile_sim":     1.0,
            "jaccard_overlap": 1.0,
            "in_training":     0,
        }
        positive = {
            "entity":     true_tail,
            "label":      1,
            "soft_label": 1.0,
            "neg_type":   "positive",
            "features":   pos_features,
            "text_input": build_text_input_codex(
                head, relation, true_tail,
                features  = pos_features,
                true_tail = None
            ),
        }

        # ── Standard negatives ────────────────────────────────────────────────
        negatives = sample_negatives_codex(
            head, relation, true_tail, kge_scores,
            n_model=n_model, n_type=n_type,
            n_intrain=n_intrain, n_random=n_random
        )

        # ── Agent wrong prediction as hard negative ───────────────────────────
        if not is_correct and agent_pred and agent_pred != true_tail:
            if agent_pred in label_to_entity_id:
                agent_score    = kge_scores.get(agent_pred, -99.0)
                agent_in_train = int(
                    agent_pred in ent_rel_to_tails.get((head, relation), [])
                )
                negatives.append({
                    "entity":   agent_pred,
                    "label":    0,
                    "neg_type": "agent_wrong",
                    "soft_label": 0.75,
                    "features": {
                        "kge_score":       agent_score,
                        "type_fit":        type_fit(agent_pred, relation),
                        "profile_sim":     round(profile_similarity(agent_pred, true_tail), 4),
                        "jaccard_overlap": round(profile_similarity(agent_pred, true_tail), 4),
                        "in_training":     agent_in_train,
                    }
                })
            stats["wrong"] += 1
        else:
            stats["correct"] += 1

        # ── Noise injection ───────────────────────────────────────────────────
        seen_entities = {n["entity"] for n in negatives} | {true_tail, head}
        noise_pool    = [
            e for e in all_entities
            if e not in seen_entities
            and profile_similarity(e, true_tail) < 0.05
        ]
        random.shuffle(noise_pool)
        for entity in noise_pool[:n_noise]:
            negatives.append({
                "entity":     entity,
                "label":      0,
                "neg_type":   "noise",
                "soft_label": 0.0,
                "features": {
                    "kge_score":       kge_scores.get(entity, -99.0),
                    "type_fit":        type_fit(entity, relation),
                    "profile_sim":     0.0,
                    "jaccard_overlap": 0.0,
                    "in_training":     0,
                }
            })
            stats["noise_added"] += 1

        # ── Add text inputs ───────────────────────────────────────────────────
        for neg in negatives:
            if "text_input" not in neg:
                neg["text_input"] = build_text_input_codex(
                    head, relation, neg["entity"],
                    features  = neg["features"],
                    true_tail = true_tail
                )

        # ── Soft labels ───────────────────────────────────────────────────────
        all_candidates = [positive] + negatives
        all_candidates = compute_soft_labels_codex(all_candidates, agent_conf)
        all_candidates.sort(key=lambda c: -c["soft_label"])

        ranked_entities = sorted(kge_scores.keys(), key=lambda e: -kge_scores[e])

        bert_records.append({
            "triple":              [head, relation, true_tail],
            "true_rank":           pre["true_rank"],
            "hop_type":            pre.get("hop_type",  "multi"),
            "hard_failure":        pre.get("hard_failure", False),
            "agent_confidence":    agent_conf,
            "is_correct":          is_correct,
            "rotate_rank_in_topk": pre["true_rank"],
            "candidates": [
                {
                    "entity":     c["entity"],
                    "label":      c["label"],
                    "soft_label": c["soft_label"],
                    "neg_type":   c.get("neg_type", "positive"),
                    "kge_rank":   ranked_entities.index(c["entity"]) + 1
                                  if c["entity"] in kge_scores else 999,
                    "text_input": c["text_input"],
                    "features":   c.get("features", {}),
                }
                for c in all_candidates
            ],
            "context": {
                "subgraph_str":  pre.get("fail_summary", ""),
                "only_tail_has": pre.get("only_tail_has", []),
                "fail_summary":  pre.get("fail_summary", ""),
                "known_tails":   pre.get("known_tails",  []),
            }
        })

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(agent_results)} done  "
                  f"({len(bert_records)} records built)")

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(output_path, "w") as f:
        json.dump(bert_records, f, indent=2)

    if not bert_records:
        print("⚠ No records built")
        return bert_records

    all_soft  = [c["soft_label"] for r in bert_records
                 for c in r["candidates"] if c["label"] == 0]
    neg_types = [c["neg_type"]   for r in bert_records
                 for c in r["candidates"] if c["label"] == 0]

    print(f"\n{'='*50}")
    print(f"Records built:     {len(bert_records)}")
    print(f"  correct:         {stats['correct']}")
    print(f"  wrong (used):    {stats['wrong']}")
    print(f"  parse fail:      {stats['parse_fail']}")
    print(f"  index miss:      {stats['index_miss']}")
    print(f"  noise injected:  {stats['noise_added']}")
    print(f"Skipped total:     {skipped}")
    print(f"Avg candidates:    "
          f"{sum(len(r['candidates']) for r in bert_records)/len(bert_records):.1f}")
    print(f"\nSoft label stats (negatives only):")
    print(f"  min  : {min(all_soft):.3f}")
    print(f"  max  : {max(all_soft):.3f}")
    print(f"  mean : {np.mean(all_soft):.3f}")
    print(f"\nNeg type breakdown:")
    for nt, count in Counter(neg_types).most_common():
        avg = np.mean([c["soft_label"] for r in bert_records
                       for c in r["candidates"]
                       if c["label"] == 0 and c["neg_type"] == nt])
        print(f"  {nt:<20} {count:>4}  avg_soft={avg:.3f}")

    return bert_records


random.seed(42)

bert_data = build_full_codex_bert_dataset(
    agent_results = agent_results,
    output_path   = "/kaggle/working/codex_bert_reranker_train.json",
    n_model       = 4,
    n_type        = 4,
    n_intrain     = 3,
    n_random      = 2,
    n_noise       = 2,
)


# ════════════════════════════════════════════════════════
# DATASET CLASS
# Identical to UMLS — drop-in replacement
# ════════════════════════════════════════════════════════

tokenizer = AutoTokenizer.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

class CoDExRerankerDataset(Dataset):
    def __init__(self, data, tokenizer,
                 max_length=MAX_LENGTH,
                 hard_k=2, rand_k=2, seed=None):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length
        self.hard_k    = hard_k
        self.rand_k    = rand_k
        self.seed      = seed
        self.pairs     = []
        self.resample()

    def resample(self):
        rng        = random.Random(self.seed)
        self.pairs = []
        for ex in self.data:
            pos = next(
                (c for c in ex["candidates"] if c["label"] == 1), None
            )
            if pos is None:
                continue
            neg_pool = [c for c in ex["candidates"] if c["label"] == 0]
            neg_pool.sort(key=lambda c: -c["soft_label"])
            selected  = neg_pool[:self.hard_k]
            remainder = neg_pool[self.hard_k:]
            if remainder and self.rand_k > 0:
                selected += rng.sample(remainder,
                                       min(self.rand_k, len(remainder)))
            for neg in selected:
                self.pairs.append({
                    "pos_text":   pos["text_input"],
                    "neg_text":   neg["text_input"],
                    "soft_label": float(neg["soft_label"]),
                    "neg_type":   neg.get("neg_type", "unknown"),
                })
        print(f"Pairs built: {len(self.pairs)}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair    = self.pairs[idx]
        pos_enc = self.tokenizer(
            pair["pos_text"],
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        neg_enc = self.tokenizer(
            pair["neg_text"],
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        return {
            "pos_input_ids":      pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids":      neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
            "soft_label":         torch.tensor(pair["soft_label"],
                                               dtype=torch.float32),
            "neg_type":           pair["neg_type"],
        }


# Verify
dataset = CoDExRerankerDataset(bert_data, tokenizer, hard_k=2, rand_k=2, seed=42)
pair    = dataset[0]
print("\nPos decoded:")
print(tokenizer.decode(pair["pos_input_ids"], skip_special_tokens=True)[:300])
print("\nNeg decoded:")
print(tokenizer.decode(pair["neg_input_ids"], skip_special_tokens=True)[:300])
print("\nSoft label:", pair["soft_label"].item())
print("Neg type:  ", pair["neg_type"])

Agent results : 101
Train records : 32888
Test records  : 1828
Index size    : 34716
Agent confidence lookup: 92 triples
  10/101 done  (10 records built)
  20/101 done  (20 records built)
  30/101 done  (30 records built)
  40/101 done  (40 records built)
  50/101 done  (50 records built)
  60/101 done  (60 records built)
  70/101 done  (70 records built)
  80/101 done  (80 records built)
  90/101 done  (90 records built)
  100/101 done  (100 records built)

Records built:     101
  correct:         84
  wrong (used):    17
  parse fail:      0
  index miss:      0
  noise injected:  202
Skipped total:     0
Avg candidates:    14.3

Soft label stats (negatives only):
  min  : 0.000
  max  : 0.890
  mean : 0.401

Neg type breakdown:
  model_topk            404  avg_soft=0.598
  type_confusable       398  avg_soft=0.436
  noise                 202  avg_soft=0.097
  random                202  avg_soft=0.000
  in_training           122  avg_soft=0.782
  agent_wrong            17  avg_soft

In [108]:
import json, torch, numpy as np
import torch.nn.functional as F
from collections import defaultdict

# ── config ────────────────────────────────────────────────────────────────────
TOPK           = 100
HELD_OUT_INPUT = "/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/CODEX_S_held_out (2).json"
HELD_OUT_OUT   = "/kaggle/working/codex_bert_held_out.json"

# ── All of these already exist from your earlier cells ───────────────────────
# label_to_entity_id, id_to_entity_label
# label_to_relation_id, id_to_relation_label
# entity_emb, relation_emb, DIM
# _scores_sp, score_triple
# ENT_RE, ENT_IM, EMBS_NORM
# constraints (from build_type_constraints)
# ent_rel_to_tails, ent_to_rels
# df_train

# ── Entity relation profile (same as UMLS version) ───────────────────────────
# ent_to_rels already built in your GPU setup cell
# use it directly — no rebuild needed

# ── EMBS for cosine similarity ────────────────────────────────────────────────
# EMBS_CACHE already built — use it directly

print(f"Entities:  {len(label_to_entity_id)}")
print(f"Relations: {len(label_to_relation_id)}")
print(f"EMBS:      {EMBS_CACHE.shape}")




@torch.no_grad()
def get_topk_candidates_codex(head, relation, true_tail, k=TOPK):
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    o = label_to_entity_id.get(true_tail)

    if None in (s, p):
        return []

    # Score all entities at once on GPU
    scores_gpu = (
        (ENT_RE[s] * relation_emb_gpu[p, :DIM] -
         ENT_IM[s] * relation_emb_gpu[p, DIM:]) @ ENT_RE.T +
        (ENT_RE[s] * relation_emb_gpu[p, DIM:] +
         ENT_IM[s] * relation_emb_gpu[p, :DIM]) @ ENT_IM.T
    )
    scores_gpu[s] = float("-inf")   # exclude self
    scores_cpu = scores_gpu.cpu().numpy()

    ranked_ids = np.argsort(scores_cpu)[::-1]

    candidates = []
    rank = 0
    for idx in ranked_ids:
        entity = id_to_entity_label[idx]
        if entity == head:
            continue
        rank += 1
        candidates.append({
            "entity":    entity,
            "kge_score": round(float(scores_cpu[idx]), 4),
            "kge_rank":  rank,
        })
        if len(candidates) == k:
            break

    # force-insert true tail if outside top-K
    if true_tail and not any(c["entity"] == true_tail for c in candidates):
        if o is not None:
            t_score = float(scores_cpu[o])
            t_rank  = int(np.sum(scores_cpu > t_score))
            candidates.append({
                "entity":    true_tail,
                "kge_score": round(t_score, 4),
                "kge_rank":  t_rank,
            })

    return candidates



def extract_features_codex(head, relation, candidate, true_tail):
    s = label_to_entity_id.get(head)
    p = label_to_relation_id.get(relation)
    c = label_to_entity_id.get(candidate)

    if None in (s, p, c):
        return {}

    # KGE score
    kge_score = score_triple(s, p, c)

    # Embedding similarity using magnitude embeddings
    h_vec = EMBS_CACHE[s]
    c_vec = EMBS_CACHE[c]
    sim_to_head = F.cosine_similarity(
        h_vec.unsqueeze(0), c_vec.unsqueeze(0)
    ).item()

    o = label_to_entity_id.get(true_tail)
    if o is not None:
        t_vec = EMBS_CACHE[o]
        sim_to_true_tail = F.cosine_similarity(
            t_vec.unsqueeze(0), c_vec.unsqueeze(0)
        ).item()
    else:
        sim_to_true_tail = 0.0

    # Relational profile overlap
    # ent_to_rels maps entity → set of relations it participates in
    true_rels = ent_to_rels.get(true_tail, set())
    cand_rels = ent_to_rels.get(candidate, set())

    shared    = true_rels & cand_rels
    only_true = true_rels - cand_rels
    only_cand = cand_rels - true_rels
    union     = true_rels | cand_rels
    jaccard   = len(shared) / len(union) if union else 0.0

    # Type constraint signals
    rel_dist  = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = rel_dist.get(candidate, 0.0)
    ranked    = list(rel_dist.keys())
    type_rank = ranked.index(candidate) + 1 if candidate in ranked else 99
    cooc      = constraints["rel_to_tail_counts"].get(
                    relation, {}).get(candidate, 0)

    # In training signal  (CoDEx-S specific — not in UMLS version)
    known_as_tail = candidate in ent_rel_to_tails.get((head, relation), [])

    return {
        # Nations/UMLS compatible keys
        "kge_score":         round(kge_score,        4),
        "sim_to_head":       round(sim_to_head,       4),
        "sim_to_true_tail":  round(sim_to_true_tail,  4),
        "shared_rels_count": len(shared),
        "only_true_count":   len(only_true),
        "only_cand_count":   len(only_cand),
        "jaccard_overlap":   round(jaccard,           4),
        "direct_edge":       int(tf > 0.0),
        "hop_count":         1,
        # CoDEx-S extras
        "type_fit":          round(tf,                4),
        "type_rank":         type_rank,
        "cooc_count":        cooc,
        "in_training":       int(known_as_tail),      # ← key CoDEx-S feature
    }



def build_text_input_codex(head, relation, candidate,
                            features, true_tail=None):
    f         = features
    sig       = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = sig.get(candidate, 0.0)
    ranked    = list(sig.keys())
    type_rank = ranked.index(candidate) + 1 if candidate in ranked else 99
    expected  = ", ".join(ranked[:3]) if ranked else "none"
    cand_rels = sorted(ent_to_rels.get(candidate, set()))[:5]

    # Known tails for this (head, relation) pair
    known = ent_rel_to_tails.get((head, relation), [])
    known_str = ", ".join(known[:5]) if known else "none"

    text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{candidate}\n\n"
        f"[TYPE CONSTRAINTS]\n"
        f"type_fit = {tf:.4f}\n"
        f"type_rank = {type_rank}\n"
        f"expected_tails = {expected}\n\n"
        f"[KGE SIGNALS]\n"
        f"kge_score = {f.get('kge_score', 0.0):.3f}\n\n"
        f"[TRAINING EVIDENCE]\n"
        f"known_tails = {known_str}\n"
        f"in_training = {f.get('in_training', 0)}\n\n"
        f"[RELATIONAL PROFILE]\n"
        f"candidate_appears_in = "
        f"{', '.join(cand_rels) if cand_rels else 'none'}\n"
    )

    return text



def convert_held_out_record_codex(record):
    head      = record["head"]
    relation  = record["relation"]
    true_tail = record["tail"]

    if head      not in label_to_entity_id:   return None
    if true_tail not in label_to_entity_id:   return None
    if relation  not in label_to_relation_id: return None

    # Get top-K candidates from ComplEx
    raw = get_topk_candidates_codex(head, relation, true_tail, k=TOPK)
    if not raw:
        return None

    # Reuse precomputed fields from preprocessed record
    only_tail_has = record.get("only_tail_has", [])
    fail_summary  = record.get("fail_summary",  "")
    sim_head_str  = record.get("sim_head",       "")
    known_tails   = record.get("known_tails",    [])

    # Subgraph string
    raw_subgraph = record.get("subgraph", [])
    oth_set      = set(only_tail_has)
    subgraph_str = "\n".join(
        f"  {h} --{r}--> {t}" + (" ◆" if r in oth_set else "")
        for triple in raw_subgraph[:12]
        for h, r, t in [triple if isinstance(triple, (list, tuple))
                         else (triple.get("head",""),
                               triple.get("relation",""),
                               triple.get("tail",""))]
    )

    # Build candidates with features + text_input
    candidates_out = []
    for c in raw:
        entity   = c["entity"]
        features = extract_features_codex(head, relation, entity, true_tail)
        if not features:
            continue
        text_input = build_text_input_codex(
            head, relation, entity,
            features  = features,
            true_tail = true_tail if entity != true_tail else None
        )
        candidates_out.append({
            "entity":     entity,
            "label":      1 if entity == true_tail else 0,
            "soft_label": None,
            "neg_type":   "true" if entity == true_tail else "model_topk",
            "kge_rank":   c["kge_rank"],
            "text_input": text_input,
            "features":   features,
        })

    # Sort by kge_score descending
    candidates_out.sort(
        key=lambda c: -(c["features"].get("kge_score") or -99)
    )

    rotate_rank = next(
        (i + 1 for i, c in enumerate(candidates_out)
         if c["entity"] == true_tail), None
    )

    return {
        "triple":              [head, relation, true_tail],
        "true_rank":           record.get("true_rank", -1),
        "hop_type":            record.get("hop_type",  "multi"),
        "hard_failure":        record.get("hard_failure", False),
        "agent_confidence":    0.5,
        "rotate_rank_in_topk": rotate_rank,
        "candidates":          candidates_out,
        "context": {
            "subgraph_str":  subgraph_str,
            "only_tail_has": only_tail_has,
            "fail_summary":  fail_summary,
            "sim_head":      sim_head_str,
            "known_tails":   known_tails,   # ← CoDEx-S extra
        }
    }



with open(HELD_OUT_INPUT) as f:
    held_out_records = json.load(f)

print(f"Total held-out: {len(held_out_records)}")

reranker_held_out, errors = [], 0

for i, record in enumerate(held_out_records):
    try:
        out = convert_held_out_record_codex(record)
        if out is not None:
            reranker_held_out.append(out)
    except Exception as e:
        print(f"[ERROR] {i}: {e}")
        errors += 1
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(held_out_records)} done")

with open(HELD_OUT_OUT, "w") as f:
    json.dump(reranker_held_out, f, indent=2)

print(f"\nOutput: {len(reranker_held_out)} records  ({errors} errors)")

# ── Sanity checks ─────────────────────────────────────────────────────────────
missing = sum(
    1 for r in reranker_held_out
    if not any(c["entity"] == r["triple"][2] for c in r["candidates"])
)
print(f"Missing true tail in candidates: {missing}")

rotate_ranks = [
    r["rotate_rank_in_topk"]
    for r in reranker_held_out
    if r["rotate_rank_in_topk"] is not None
]
hits1 = sum(1 for r in rotate_ranks if r == 1)
hits3 = sum(1 for r in rotate_ranks if r <= 3)

print(f"\nComplEx baseline within top-{TOPK}:")
print(f"  Hits@1: {hits1}/{len(rotate_ranks)}"
      f"  ({100*hits1/max(len(rotate_ranks),1):.1f}%)")
print(f"  Hits@3: {hits3}/{len(rotate_ranks)}"
      f"  ({100*hits3/max(len(rotate_ranks),1):.1f}%)")

# Type constraint coverage
has_type_fit  = sum(
    1 for r in reranker_held_out
    for c in r["candidates"]
    if c["features"].get("type_fit", 0) > 0
)
in_training   = sum(
    1 for r in reranker_held_out
    for c in r["candidates"]
    if c["features"].get("in_training", 0) == 1
)
total_cands   = sum(len(r["candidates"]) for r in reranker_held_out)

print(f"\nFeature coverage:")
print(f"  type_fit > 0:  {has_type_fit}/{total_cands}"
      f"  ({100*has_type_fit/max(total_cands,1):.1f}%)")
print(f"  in_training=1: {in_training}/{total_cands}"
      f"  ({100*in_training/max(total_cands,1):.1f}%)")

print(f"\nSample positive text_input:")
pos = next(c for c in reranker_held_out[0]["candidates"] if c["label"] == 1)
print(pos["text_input"])

print(f"\nSample negative text_input:")
neg = next(c for c in reranker_held_out[0]["candidates"] if c["label"] == 0)
print(neg["text_input"])

print(f"\nSaved → {HELD_OUT_OUT}")

Entities:  2034
Relations: 42
EMBS:      torch.Size([2034, 256])
Total held-out: 914
  25/914 done
  50/914 done
  75/914 done
  100/914 done
  125/914 done
  150/914 done
  175/914 done
  200/914 done
  225/914 done
  250/914 done
  275/914 done
  300/914 done
  325/914 done
  350/914 done
  375/914 done
  400/914 done
  425/914 done
  450/914 done
  475/914 done
  500/914 done
  525/914 done
  550/914 done
  575/914 done
  600/914 done
  625/914 done
  650/914 done
  675/914 done
  700/914 done
  725/914 done
  750/914 done
  775/914 done
  800/914 done
  825/914 done
  850/914 done
  875/914 done
  900/914 done

Output: 914 records  (0 errors)
Missing true tail in candidates: 0

ComplEx baseline within top-100:
  Hits@1: 114/914  (12.5%)
  Hits@3: 215/914  (23.5%)

Feature coverage:
  type_fit > 0:  55573/91434  (60.8%)
  in_training=1: 11271/91434  (12.3%)

Sample positive text_input:
[QUERY]
Samoa | member of | ?

[CANDIDATE]
"African, Caribbean and Pacific Group of States"

[TYPE

In [109]:
total         = len(agent_results)
correct       = sum(1 for r in agent_results if r.get("final_correct"))
wrong         = sum(1 for r in agent_results if not r.get("final_correct"))
resolved      = sum(1 for r in agent_results if r.get("aggregator",{}).get("failure_type")=="resolved")
both_failed   = sum(1 for r in agent_results if r.get("aggregator",{}).get("failure_type")=="both_failed")
correct_but_not_resolved = sum(1 for r in agent_results 
                                if r.get("final_correct") 
                                and r.get("aggregator",{}).get("failure_type")!="resolved")

print(f"Total:                      {total}")
print(f"Correct predictions:        {correct}")
print(f"Wrong predictions:          {wrong}")
print(f"failure_type=resolved:      {resolved}")
print(f"failure_type=both_failed:   {both_failed}")
print(f"Correct but not resolved:   {correct_but_not_resolved}")

Total:                      101
Correct predictions:        84
Wrong predictions:          17
failure_type=resolved:      89
failure_type=both_failed:   1
Correct but not resolved:   10


In [110]:
# Find the 19 missing records
built_triples = set()
for result in agent_results:
    if result.get("final_correct"):
        raw   = result["triple"].strip("()")
        parts = [p.strip() for p in raw.split(",")]
        if len(parts) == 3:
            h, r, t = parts
            if preprocessed_index.get((h, r, t)):
                built_triples.add(result["triple"])

print(f"Correct:        84")
print(f"Built:          {len(built_triples)}")
print(f"Missing:        {84 - len(built_triples)}")

# Show the missing ones
for result in agent_results:
    if result.get("final_correct") and result["triple"] not in built_triples:
        print(f"  MISSING: {result['triple']}")

Correct:        84
Built:          75
Missing:        9
  MISSING: (Angela Merkel, languages spoken, written, or signed, German)
  MISSING: (Tamara Bunke, languages spoken, written, or signed, German)
  MISSING: (Omar Sharif, languages spoken, written, or signed, Arabic)
  MISSING: (Mircea Eliade, languages spoken, written, or signed, English)
  MISSING: (Marlene Dietrich, languages spoken, written, or signed, French)
  MISSING: (Romain Gary, languages spoken, written, or signed, Polish)
  MISSING: (Yuri Shevchuk, languages spoken, written, or signed, Russian)
  MISSING: (Albert Einstein, languages spoken, written, or signed, English)
  MISSING: (George Frideric Handel, languages spoken, written, or signed, German)


In [113]:
# ════════════════════════════════════════════════════════
# AGENT C — BGE RERANKER TRAINING (MERGED)
# Adaptive margin loss + staged unfreezing + resampling
# ════════════════════════════════════════════════════════

import json, os, random, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from collections import defaultdict
import numpy as np

# ── config ────────────────────────────────────────────────
MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# Try in order:
# 1. "BAAI/bge-reranker-base"                    — best, may need HF token
# 2. "cross-encoder/ms-marco-MiniLM-L-6-v2"     — always works, no auth
# 3. "cross-encoder/ms-marco-MiniLM-L-2-v2"     — lightest

MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS     = 5
LR         = 2e-5
MARGIN     = 1.0
SAVE_DIR   = "agent_c_bge"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ════════════════════════════════════════════════════════
# MODEL LOADING
# ════════════════════════════════════════════════════════

def load_model_and_tokenizer(model_name: str):
    """
    Tries BAAI first, falls back gracefully.
    BAAI/bge-reranker-base sometimes needs:
      pip install -U FlagEmbedding
    or HF login:
      huggingface-cli login
    """
    try:
        print(f"Loading {model_name}...")
        tok = AutoTokenizer.from_pretrained(
            model_name, trust_remote_code=True
        )
        mdl = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1,
            trust_remote_code=True,
            ignore_mismatched_sizes=True,
        ).to(DEVICE)
        print(f"Loaded: {model_name}")
        return mdl, tok

    except Exception as e:
        fallback = "cross-encoder/ms-marco-MiniLM-L-6-v2"
        print(f"[WARN] {model_name} failed: {e}")
        print(f"[WARN] Falling back to {fallback}")
        tok = AutoTokenizer.from_pretrained(fallback)
        mdl = AutoModelForSequenceClassification.from_pretrained(
            fallback, num_labels=1
        ).to(DEVICE)
        return mdl, tok


# ════════════════════════════════════════════════════════
# DATASET — supports resampling for staged training
#
# hard_k  : top-k hardest negatives by soft_label (descending)
# rand_k  : additional random negatives for diversity
# seed    : fixed seed locks val set; None = fresh draw each resample()
# ════════════════════════════════════════════════════════

class BGERerankerDataset(Dataset):

    def __init__(self, data: list, tokenizer,
                 max_length: int = MAX_LENGTH,
                 hard_k: int = 2, rand_k: int = 2,
                 seed: int | None = None):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length
        self.hard_k    = hard_k
        self.rand_k    = rand_k
        self.seed      = seed
        self.pairs     = []
        self.resample()

    def resample(self):
        """Rebuild pairs — call once per epoch on train_ds for fresh negatives."""
        rng = random.Random(self.seed)
        self.pairs = []

        for ex in self.data:
            pos = next(
                (c for c in ex["candidates"] if c["label"] == 1), None
            )
            if pos is None:
                continue

            neg_pool = [c for c in ex["candidates"] if c["label"] == 0]
            neg_pool.sort(key=lambda c: -c["soft_label"])

            selected = neg_pool[:self.hard_k]
            remainder = neg_pool[self.hard_k:]
            if remainder and self.rand_k > 0:
                k = min(self.rand_k, len(remainder))
                selected += rng.sample(remainder, k)

            for neg in selected:
                self.pairs.append({
                    "pos_text":   pos["text_input"],
                    "neg_text":   neg["text_input"],
                    "soft_label": float(neg["soft_label"]),
                    "neg_type":   neg.get("neg_type", "unknown"),
                    "true_rank":  ex.get("true_rank", 99),
                })

        print(f"Pairs built: {len(self.pairs)}")
        self._print_stats()

    def _print_stats(self):
        by_type = defaultdict(list)
        for p in self.pairs:
            by_type[p["neg_type"]].append(p["soft_label"])
        print("  Negative breakdown:")
        for nt, labels in sorted(by_type.items()):
            print(f"    {nt:<12} n={len(labels):>4}  "
                  f"avg_soft={np.mean(labels):.3f}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int) -> dict:
        pair    = self.pairs[idx]
        pos_enc = self.tokenizer(
            pair["pos_text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        neg_enc = self.tokenizer(
            pair["neg_text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "pos_input_ids":      pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids":      neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
            "soft_label":         torch.tensor(pair["soft_label"],
                                               dtype=torch.float32),
            "neg_type":           pair["neg_type"],
        }


# ════════════════════════════════════════════════════════
# LOSS — adaptive margin + correct weighting + MSE aux
#
# FIX 1 — adaptive margin
#   margin shrinks as soft_label rises
#   soft=0.85 (hard/ambiguous) → margin=0.15  gentle push
#   soft=0.00 (easy)           → margin=1.0   full push
#
# FIX 2 — correct weighting
#   weight = 1 + (1 - soft_label)
#   easy negatives (soft=0)    → weight=2.0   push hard
#   hard negatives (soft=0.85) → weight=1.15  push gently
#
# FIX 3 — MSE auxiliary loss
#   pred_diff = score_pos - score_neg should equal (1 - soft)
#   anchors calibration so scores are meaningful, not just ordinal
#
# Doc-1 alias: FullDistillationLoss → same contract, kept as AdaptiveSoftLoss
# ════════════════════════════════════════════════════════

class AdaptiveSoftLoss(nn.Module):
    """
    Replaces FullDistillationLoss from Doc-1.
    Signature-compatible: forward() returns (total, pairwise, mse).
    Doc-1 used w_mse / w_abs; here mse_weight covers both.
    """

    def __init__(self, margin: float = MARGIN, mse_weight: float = 0.3,
                 # Accept but ignore Doc-1 kwargs for drop-in compatibility
                 w_mse: float | None = None, w_abs: float | None = None):
        super().__init__()
        self.margin     = margin
        # If called with Doc-1's w_mse, honour it
        self.mse_weight = w_mse if w_mse is not None else mse_weight

    def forward(self,
                score_pos:   torch.Tensor,
                score_neg:   torch.Tensor,
                soft_labels: torch.Tensor,
                ) -> tuple[torch.Tensor, float, float]:

        adaptive_margin = (1.0 - soft_labels) * self.margin
        raw_loss = torch.clamp(
            adaptive_margin - (score_pos - score_neg), min=0.0
        )
        weights       = 1.0 + (1.0 - soft_labels)
        pairwise_loss = (raw_loss * weights).mean()

        pred_diff   = score_pos - score_neg
        target_diff = 1.0 - soft_labels
        mse_loss    = nn.functional.mse_loss(pred_diff, target_diff)

        total = pairwise_loss + self.mse_weight * mse_loss
        return total, pairwise_loss.item(), mse_loss.item()

# Alias so old imports / references still resolve
FullDistillationLoss = AdaptiveSoftLoss


# ════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════

def compute_metrics(score_pos: torch.Tensor,
                    score_neg: torch.Tensor,
                    neg_types: list) -> dict:
    correct = (score_pos > score_neg).cpu()
    overall = correct.float().mean().item()

    by_type = defaultdict(lambda: {"correct": 0, "total": 0})
    for c, nt in zip(correct.tolist(), neg_types):
        by_type[nt]["correct"] += int(c)
        by_type[nt]["total"]   += 1

    return {
        "overall": overall,
        "by_type": {
            nt: v["correct"] / v["total"]
            for nt, v in by_type.items()
        },
    }


# ════════════════════════════════════════════════════════
# TRAIN EPOCH
# ════════════════════════════════════════════════════════

def train_epoch(model, loader, optimizer,
                scheduler, loss_fn, device) -> dict:
    model.train()
    total_loss = total_pair = total_mse = 0.0
    all_pos, all_neg, types = [], [], []

    for batch in loader:
        pos_ids  = batch["pos_input_ids"].to(device)
        pos_mask = batch["pos_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        soft     = batch["soft_label"].to(device)

        optimizer.zero_grad()
        score_pos = model(input_ids=pos_ids,
                          attention_mask=pos_mask).logits.squeeze(-1)
        score_neg = model(input_ids=neg_ids,
                          attention_mask=neg_mask).logits.squeeze(-1)

        loss, pair_l, mse_l = loss_fn(score_pos, score_neg, soft)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_pair += pair_l
        total_mse  += mse_l
        all_pos.extend(score_pos.detach().cpu().tolist())
        all_neg.extend(score_neg.detach().cpu().tolist())
        types.extend(batch["neg_type"])

    n = len(loader)
    metrics = compute_metrics(
        torch.tensor(all_pos), torch.tensor(all_neg), types
    )
    metrics["loss"]      = round(total_loss / n, 4)
    metrics["pairwise"]  = round(total_pair / n, 4)
    metrics["mse"]       = round(total_mse  / n, 4)
    metrics["abs"]       = metrics["mse"]   # kept for Doc-1 print compat
    return metrics


# ════════════════════════════════════════════════════════
# EVAL EPOCH
# ════════════════════════════════════════════════════════

@torch.no_grad()
def eval_epoch(model, loader, loss_fn, device) -> dict:
    model.eval()
    total_loss = total_pair = total_mse = 0.0
    all_pos, all_neg, types = [], [], []

    for batch in loader:
        pos_ids  = batch["pos_input_ids"].to(device)
        pos_mask = batch["pos_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        soft     = batch["soft_label"].to(device)

        score_pos = model(input_ids=pos_ids,
                          attention_mask=pos_mask).logits.squeeze(-1)
        score_neg = model(input_ids=neg_ids,
                          attention_mask=neg_mask).logits.squeeze(-1)

        loss, pair_l, mse_l = loss_fn(score_pos, score_neg, soft)
        total_loss += loss.item()
        total_pair += pair_l
        total_mse  += mse_l
        all_pos.extend(score_pos.cpu().tolist())
        all_neg.extend(score_neg.cpu().tolist())
        types.extend(batch["neg_type"])

    n = len(loader)
    metrics = compute_metrics(
        torch.tensor(all_pos), torch.tensor(all_neg), types
    )
    metrics["loss"]     = round(total_loss / n, 4)
    metrics["pairwise"] = round(total_pair / n, 4)
    metrics["mse"]      = round(total_mse  / n, 4)
    metrics["abs"]      = metrics["mse"]
    return metrics


# ════════════════════════════════════════════════════════
# TRAIN — staged unfreezing + per-epoch resampling
#
# Healthy convergence targets:
#   epoch 1 (head only):  train acc 0.75-0.85  val acc 0.85-0.92
#   epoch 2 (unfrozen):   train/val acc rising  mse still falling
#   epoch 3+:             train ~0.92           val ~0.88-0.92  mse < 0.15
#
# RED FLAGS:
#   val acc 0.99 in epoch 1  → val too easy
#   mse rising, pair falling → score gap overshooting
#   train acc >> val acc     → overfit on hard negatives
# ════════════════════════════════════════════════════════

def train(data_path: str = "/kaggle/working/codex_bert_reranker_train.json"):

    with open(data_path) as f:
        raw = json.load(f)
    print(f"Records loaded: {len(raw)}")

    random.seed(42)
    random.shuffle(raw)

    # Fix 1 — increase val to 20% (~20 records, ~80 pairs)
    n_val     = max(3, int(len(raw) * 0.20))
    val_raw   = raw[:n_val]
    train_raw = raw[n_val:]
    print(f"Train: {len(train_raw)}  Val: {len(val_raw)}")

    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)

    train_ds = BGERerankerDataset(
        train_raw, tokenizer,
        hard_k=2, rand_k=2, seed=None
    )

    # Fix 2 — val uses ALL negatives (hard_k=5) not just 2+2
    # More pairs per record = harder to memorize
    val_ds = BGERerankerDataset(
        val_raw, tokenizer,
        hard_k=5, rand_k=0,   # ← all hard, no random padding
        seed=42
    )

    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True,
    )

    # Fix 3 — reduce MSE weight, it dominated in epoch 1
    # mse=0.92 vs pair=0.81 means MSE is driving everything
    loss_fn = AdaptiveSoftLoss(margin=MARGIN, w_mse=0.2)

    os.makedirs(SAVE_DIR, exist_ok=True)
    best_val_loss = float("inf")
    best_val_acc  = 0.0
    history       = []
    optimizer     = None

    for epoch in range(1, EPOCHS + 1):

        if epoch == 1:
            for param in model.base_model.parameters():
                param.requires_grad = False
            print(f"\n── Epoch {epoch}: BERT frozen")
            optimizer = AdamW(
                model.classifier.parameters(),
                lr=LR * 10, weight_decay=0.01,
            )
        elif epoch == 2:
            for param in model.base_model.parameters():
                param.requires_grad = True
            print(f"\n── Epoch {epoch}: BERT unfrozen")
            optimizer = AdamW([
                {"params": model.base_model.parameters(),  "lr": LR},
                {"params": model.classifier.parameters(),  "lr": LR * 5},
            ], weight_decay=0.01)
        else:
            print(f"\n── Epoch {epoch}")

        train_ds.resample()
        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE,
            shuffle=True, num_workers=2, pin_memory=True,
        )

        total_steps = len(train_loader) * (EPOCHS - epoch + 1)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = max(1, int(total_steps * 0.1)),
            num_training_steps = total_steps,
        )

        tr = train_epoch(model, train_loader, optimizer,
                         scheduler, loss_fn, DEVICE)
        vl = eval_epoch(model, val_loader, loss_fn, DEVICE)

        history.append({"epoch": epoch, "train": tr, "val": vl})

        print(f"  train  loss={tr['loss']}  pair={tr['pairwise']}  "
              f"mse={tr['mse']}  acc={tr['overall']:.3f}")
        print(f"  val    loss={vl['loss']}  pair={vl['pairwise']}  "
              f"mse={vl['mse']}  acc={vl['overall']:.3f}")
        print(f"  val breakdown:")
        for nt, acc in sorted(vl["by_type"].items()):
            bar = "█" * int(acc * 20)
            print(f"    {nt:<20} {acc:.3f}  {bar}")

        # Save on val loss not val acc — more informative with small val set
        if vl["loss"] < best_val_loss:
            best_val_loss = vl["loss"]
            best_val_acc  = vl["overall"]
            model.save_pretrained(SAVE_DIR)
            tokenizer.save_pretrained(SAVE_DIR)
            print(f"  ✓ saved (val_loss={best_val_loss:.4f}  "
                  f"val_acc={best_val_acc:.3f})")

    with open(os.path.join(SAVE_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nDone. Best val_loss={best_val_loss:.4f} → {SAVE_DIR}/")
    return model, tokenizer, history


model, tokenizer, history = train(
    "/kaggle/working/codex_bert_reranker_train.json"
)

Records loaded: 101
Train: 81  Val: 20
Loading cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Pairs built: 324
  Negative breakdown:
    agent_wrong  n=   5  avg_soft=0.545
    in_training  n=  74  avg_soft=0.793
    model_topk   n= 118  avg_soft=0.638
    noise        n=  34  avg_soft=0.092
    random       n=  25  avg_soft=0.000
    type_confusable n=  68  avg_soft=0.455
Pairs built: 100
  Negative breakdown:
    agent_wrong  n=   1  avg_soft=0.586
    in_training  n=  26  avg_soft=0.795
    model_topk   n=  57  avg_soft=0.598
    type_confusable n=  16  avg_soft=0.516

── Epoch 1: BERT frozen
Pairs built: 324
  Negative breakdown:
    agent_wrong  n=   4  avg_soft=0.547
    in_training  n=  73  avg_soft=0.795
    model_topk   n= 128  avg_soft=0.625
    noise        n=  33  avg_soft=0.089
    random       n=  27  avg_soft=0.000
    type_confusable n=  59  avg_soft=0.450
  train  loss=1.1907  pair=0.9403  mse=1.252  acc=0.522
  val    loss=1.3101  pair=1.0618  mse=1.2412  acc=0.340
  val breakdown:
    agent_wrong          1.000  ██

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  ✓ saved (val_loss=0.5121  val_acc=0.540)

── Epoch 3
Pairs built: 324
  Negative breakdown:
    agent_wrong  n=   1  avg_soft=0.729
    in_training  n=  72  avg_soft=0.797
    model_topk   n= 133  avg_soft=0.626
    noise        n=  27  avg_soft=0.115
    random       n=  20  avg_soft=0.000
    type_confusable n=  71  avg_soft=0.453
  train  loss=0.6401  pair=0.5785  mse=0.3081  acc=0.636
  val    loss=0.3401  pair=0.318  mse=0.1106  acc=0.740
  val breakdown:
    agent_wrong          1.000  ████████████████████
    in_training          0.692  █████████████
    model_topk           0.754  ███████████████
    type_confusable      0.750  ███████████████


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ saved (val_loss=0.3401  val_acc=0.740)

── Epoch 4
Pairs built: 324
  Negative breakdown:
    agent_wrong  n=   5  avg_soft=0.557
    in_training  n=  75  avg_soft=0.793
    model_topk   n= 124  avg_soft=0.635
    noise        n=  28  avg_soft=0.112
    random       n=  35  avg_soft=0.000
    type_confusable n=  57  avg_soft=0.459
  train  loss=0.4668  pair=0.3882  mse=0.3927  acc=0.762
  val    loss=0.2681  pair=0.2168  mse=0.2568  acc=0.870
  val breakdown:
    agent_wrong          1.000  ████████████████████
    in_training          0.885  █████████████████
    model_topk           0.877  █████████████████
    type_confusable      0.812  ████████████████


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ saved (val_loss=0.2681  val_acc=0.870)

── Epoch 5
Pairs built: 324
  Negative breakdown:
    agent_wrong  n=   2  avg_soft=0.599
    in_training  n=  71  avg_soft=0.799
    model_topk   n= 124  avg_soft=0.626
    noise        n=  29  avg_soft=0.101
    random       n=  27  avg_soft=0.000
    type_confusable n=  71  avg_soft=0.439
  train  loss=0.3904  pair=0.3149  mse=0.377  acc=0.809
  val    loss=0.2745  pair=0.2125  mse=0.3101  acc=0.890
  val breakdown:
    agent_wrong          1.000  ████████████████████
    in_training          0.962  ███████████████████
    model_topk           0.877  █████████████████
    type_confusable      0.812  ████████████████

Done. Best val_loss=0.2681 → agent_c_bge/


In [114]:
import json, torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import defaultdict
from tqdm import tqdm

# ── config ────────────────────────────────────────────────
RERANKER_DIR = "agent_c_bge"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH   = 512
BATCH_SIZE   = 32
ALPHA        = 0.4
DELTA        = 0.6
T            = 3


# ════════════════════════════════════════════════════════
# LOAD MODEL
# ════════════════════════════════════════════════════════

def load_reranker(path):
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        path, trust_remote_code=True
    ).to(DEVICE)
    mdl.eval()
    return mdl, tok


# ════════════════════════════════════════════════════════
# BERT SCORING (BATCHED)
# ════════════════════════════════════════════════════════

@torch.no_grad()
def bert_score_batch(model, tokenizer, texts):
    scores = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)
        logits = model(**enc).logits.squeeze(-1)
        scores.extend(logits.cpu().tolist())
    return scores


# ════════════════════════════════════════════════════════
# NORMALISATION
# ════════════════════════════════════════════════════════

def minmax(x):
    x = np.array(x, dtype=float)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-9:
        return np.full_like(x, 0.5)
    return (x - lo) / (hi - lo)


# ════════════════════════════════════════════════════════
# INJECT TRUE TAIL IF MISSING FROM CANDIDATES
# ════════════════════════════════════════════════════════

def ensure_true_tail(candidates, true_tail, head, relation):
    if true_tail in [c["entity"] for c in candidates]:
        return candidates, False

    worst_kge = min(
        c["features"]["kge_score"]
        for c in candidates
        if c.get("features", {}).get("kge_score") is not None
    )
    injected_text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{true_tail}\n\n"
        f"[KEY SIGNALS]\n(injected — not in top-K)"
    )
    candidates = candidates + [{
        "entity":     true_tail,
        "label":      1,
        "neg_type":   "true_tail_injected",
        "kge_rank":   len(candidates) + 1,
        "text_input": injected_text,
        "features":   {"kge_score": worst_kge - 0.5},
    }]
    return candidates, True


# ════════════════════════════════════════════════════════
# MAIN EVALUATION
# ════════════════════════════════════════════════════════

def evaluate(test_path, alpha=ALPHA, delta=DELTA, T=T):

    print("Loading reranker...")
    model, tokenizer = load_reranker(RERANKER_DIR)

    with open(test_path) as f:
        data = json.load(f)
    print(f"Queries: {len(data)}")

    results    = []
    n_injected = 0
    skipped    = 0

    for rec in tqdm(data):

        triple = rec["triple"]
        head, relation, tail = (
            [x.strip() for x in triple]
            if isinstance(triple, list)
            else [x.strip() for x in triple.split("|")]
        )
        true_tail = tail

        candidates = rec.get("candidates", [])
        if not candidates:
            skipped += 1
            continue

        candidates, injected = ensure_true_tail(
            candidates, true_tail, head, relation
        )
        if injected:
            n_injected += 1

        texts = [c["text_input"] for c in candidates]

        bert_scores = bert_score_batch(model, tokenizer, texts)

        kge_scores = [
            c["features"].get("kge_score") or (1.0 / max(c.get("kge_rank", 99), 1))
            for c in candidates
        ]

        kge_norm  = minmax(kge_scores)
        bert_norm = minmax(bert_scores)
        combined  = alpha * kge_norm + delta * bert_norm

        enriched = [
            {
                "entity":     c["entity"],
                "kge_score":  ks,
                "bert_score": bs,
                "combined":   float(cs),
                "neg_type":   c.get("neg_type", "unknown"),
            }
            for c, ks, bs, cs in zip(candidates, kge_scores, bert_scores, combined)
        ]

        ranked_comb = sorted(enriched, key=lambda x: -x["combined"])
        ranked_bert = sorted(enriched, key=lambda x: -x["bert_score"])
        ranked_kge  = sorted(enriched, key=lambda x: -x["kge_score"])

        def find_rank(ranked):
            for i, x in enumerate(ranked, 1):
                if x["entity"] == true_tail:
                    return i
            return len(ranked)

        rank_comb = find_rank(ranked_comb)
        rank_bert = find_rank(ranked_bert)
        rank_kge  = find_rank(ranked_kge)

        results.append({
            "triple":       f"{head}|{relation}|{true_tail}",
            "true_tail":    true_tail,
            "true_rank":    rec.get("true_rank"),
            "rotate_rank":  rec.get("rotate_rank_in_topk"),
            "hop_type":     rec.get("hop_type", "?"),
            "hard_failure": rec.get("hard_failure", False),
            "injected":     injected,
            "rank_kge":     rank_kge,
            "rank_bert":    rank_bert,
            "rank_comb":    rank_comb,
            "top3_comb":    [x["entity"] for x in ranked_comb[:3]],
        })

    if skipped:
        print(f"[WARN] skipped {skipped} records with no candidates")
    if n_injected:
        print(f"[INFO] injected true tail into {n_injected}/{len(data)} queries")

    # ════════════════════════════════════════════════════
    # METRICS
    # ════════════════════════════════════════════════════

    def compute(ranks):
        ranks = np.array(ranks)
        return {
            "Hits@1":  float(np.mean(ranks == 1)),
            "Hits@3":  float(np.mean(ranks <= 3)),
            "Hits@10": float(np.mean(ranks <= 10)),
            "MRR":     float(np.mean(1.0 / ranks)),
        }

    def print_metrics(label, m):
        print(f"\n{label}")
        for k in ["Hits@1", "Hits@3", "Hits@10", "MRR"]:
            print(f"  {k:<8}: {m[k]:.4f}")

    kge_m  = compute([r["rank_kge"]  for r in results])
    bert_m = compute([r["rank_bert"] for r in results])
    comb_m = compute([r["rank_comb"] for r in results])

    # ── lucky rate ────────────────────────────────────────
    hits1    = [r for r in results if r["rank_comb"] == 1]
    grounded = [r for r in hits1   if r["rank_bert"] <= T]
    lucky    = [r for r in hits1   if r["rank_bert"] >  T]
    n        = len(hits1)

    # ── by hop type ───────────────────────────────────────
    by_hop = defaultdict(list)
    for r in results:
        by_hop[r["hop_type"]].append(r["rank_comb"])

    # ── hard failure recovery ─────────────────────────────
    hard     = [r for r in results if r["hard_failure"]]
    not_hard = [r for r in results if not r["hard_failure"]]

    # ── vs rotate rank ────────────────────────────────────
    improved  = sum(1 for r in results if r["rotate_rank"] is not None and r["rank_comb"] < r["rotate_rank"])
    worsened  = sum(1 for r in results if r["rotate_rank"] is not None and r["rank_comb"] > r["rotate_rank"])
    unchanged = sum(1 for r in results if r["rotate_rank"] is not None and r["rank_comb"] == r["rotate_rank"])

    # ════════════════════════════════════════════════════
    # PRINT
    # ════════════════════════════════════════════════════

    print("\n═══════════════════════════════════════════")
    print("FINAL RESULTS")
    print("═══════════════════════════════════════════")

    print_metrics("KGE  (RotatE baseline)", kge_m)
    print_metrics("BERT (reranker only)",   bert_m)
    print_metrics("COMBINED",               comb_m)

    print(f"\n── Lucky rate  (T={T}) ─────────────────────")
    if n:
        print(f"  Rank-1 queries : {n}")
        print(f"  Grounded       : {len(grounded):>4}  ({len(grounded)/n:.1%})  ← BERT agreed independently")
        print(f"  Lucky          : {len(lucky):>4}  ({len(lucky)/n:.1%})  ← KGE rescued it")
        verdict = "reranker drives wins ✓" if len(grounded) > len(lucky) else "KGE doing the work — reranker marginal"
        print(f"  Verdict        : {verdict}")
    else:
        print("  No Hits@1")

    print(f"\n── vs RotatE rank in top-K ─────────────────")
    print(f"  Improved  : {improved}")
    print(f"  Unchanged : {unchanged}")
    print(f"  Worsened  : {worsened}")

    print(f"\n── By hop type ─────────────────────────────")
    for ht, ranks in sorted(by_hop.items()):
        m = compute(ranks)
        bar = "█" * int(m["Hits@1"] * 20)
        print(f"  {ht:<8}  n={len(ranks):>4}  Hits@1={m['Hits@1']:.3f}  MRR={m['MRR']:.3f}  {bar}")

    if hard:
        hm = compute([r["rank_comb"] for r in hard])
        nm = compute([r["rank_comb"] for r in not_hard])
        print(f"\n── Hard failure recovery ────────────────────")
        print(f"  hard=True   n={len(hard):>4}  Hits@1={hm['Hits@1']:.3f}  MRR={hm['MRR']:.3f}")
        print(f"  hard=False  n={len(not_hard):>4}  Hits@1={nm['Hits@1']:.3f}  MRR={nm['MRR']:.3f}")

    # ════════════════════════════════════════════════════
    # SAVE
    # ════════════════════════════════════════════════════

    output = {
        "config":  {"alpha": alpha, "delta": delta, "T": T},
        "metrics": {"kge": kge_m, "bert": bert_m, "combined": comb_m},
        "lucky": {
            "grounded":    len(grounded),
            "lucky":       len(lucky),
            "total_hits1": n,
            "lucky_rate":  len(lucky)    / n if n else 0.0,
            "ground_rate": len(grounded) / n if n else 0.0,
        },
        "vs_rotate": {
            "improved": improved, "unchanged": unchanged, "worsened": worsened
        },
        "per_query": results,
    }
    with open("inference_results.json", "w") as f:
        json.dump(output, f, indent=2)
    print("\nSaved → inference_results.json")
    return output


# ════════════════════════════════════════════════════════
if __name__ == "__main__":
    evaluate(r"/kaggle/input/datasets/aaryaupi/codex-s-val-dataset/codex_bert_held_out (1).json")

Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Queries: 914


100%|██████████| 914/914 [00:58<00:00, 15.63it/s]


═══════════════════════════════════════════
FINAL RESULTS
═══════════════════════════════════════════

KGE  (RotatE baseline)
  Hits@1  : 0.1247
  Hits@3  : 0.2352
  Hits@10 : 0.5120
  MRR     : 0.2382

BERT (reranker only)
  Hits@1  : 0.3632
  Hits@3  : 0.5755
  Hits@10 : 0.7998
  MRR     : 0.5062

COMBINED
  Hits@1  : 0.3993
  Hits@3  : 0.6138
  Hits@10 : 0.8282
  MRR     : 0.5381

── Lucky rate  (T=3) ─────────────────────
  Rank-1 queries : 365
  Grounded       :  341  (93.4%)  ← BERT agreed independently
  Lucky          :   24  (6.6%)  ← KGE rescued it
  Verdict        : reranker drives wins ✓

── vs RotatE rank in top-K ─────────────────
  Improved  : 685
  Unchanged : 130
  Worsened  : 99

── By hop type ─────────────────────────────
  multi     n= 399  Hits@1=0.481  MRR=0.605  █████████
  none      n= 515  Hits@1=0.336  MRR=0.486  ██████

── Hard failure recovery ────────────────────
  hard=True   n=  95  Hits@1=0.558  MRR=0.705
  hard=False  n= 819  Hits@1=0.381  MRR=0.519

